# LEGO Persistent-Grain Spatial Memory Agent — Reproducible / Fixed POC

This notebook grounds a tool-using LLM agent directly in the supplied LEGO 3D Gaussian Splatting PLY.

Key guarantees:

- persistent identity comes only from `grain_id`
- 3DGS opacity is decoded with `sigmoid(opacity_logit)`
- 3DGS scale is decoded with `exp(scale_log)`
- raw SH / opacity-logit / scale-log values are preserved for round-trip PLY writing
- semantic labels are never fabricated
- synthetic damage is automatically generated from the supplied baseline PLY and loaded as `latest`
- temporal comparison is exposed through `compare_sessions`
- `detect_damage` is a real class method, not a runtime monkey-patch
- damage zones cluster completely missing persistent IDs; partial Gaussian loss is reported separately
- deterministic tool validation is separate from live LLM evaluation
- live LLM evaluation does not force the expected tool or construct the answer
- rate-limit/provider failures are reported as `UNAVAILABLE`, not scored as agent failures

 Problem & Architecture

This project builds a **grounded, tool-using LLM agent for 3D Gaussian Splatting (3DGS) analysis**. Persistent `grain_id` values from the PLY provide a stable identity layer for spatial queries and baseline/latest comparison.

### Architecture

```text
3DGS PLY
   ↓
PLY parsing & 3DGS parameter decoding
   ↓
Persistent spatial memory (`grain_id`)
   ↓
Tool layer
   ├── Point / grain queries
   ├── Scene summary
   ├── Region hypotheses
   ├── Persistent-ID comparison
   ├── ΔH surface-change analysis
   └── Synthetic damage detection
   ↓
LLM Agent
   ↓
Evidence-grounded response
   ↓
Evaluation & validation

## 1. Install / configuration

Set `GROQ_API_KEY` in Colab Secrets or the environment. The notebook does not hard-code an API key.

In [1]:
!pip install -q groq scipy numpy pandas matplotlib plyfile scikit-learn

import os
import json
import time
import math
import re
import copy
from collections import Counter
from dataclasses import dataclass
from typing import Optional

import numpy as np
import pandas as pd
from scipy.spatial import KDTree, cKDTree
from sklearn.cluster import DBSCAN
from groq import Groq

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY3")
except Exception:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY3", "")

MODEL = os.environ.get("SPATIAL_AGENT_MODEL", "openai/gpt-oss-120b")
PLY_PATH = os.environ.get("LEGO_PLY_PATH", "point_cloud (7).ply")
CONFIG_PATH = os.environ.get("LEGO_AGENT_CONFIG", "lego_persistent_grain_agent_config.json")
SYNTHETIC_PLY_PATH = os.environ.get("LEGO_SYNTHETIC_PLY", "latest_damaged_scene.ply")

# Live evaluation defaults. Full-tool evaluation is intentionally separate from routed runtime.
RUN_LIVE_LLM_EVAL = True
RUN_NEGATIVE_LLM_EVAL = False
EVAL_MAX_TURNS = 4
EVAL_MAX_TOKENS = 420
EVAL_RETRIES_429 = 2
EVAL_BACKOFF_SECONDS = 2.5

if not GROQ_API_KEY:
    raise ValueError("Set GROQ_API_KEY in Colab Secrets or as an environment variable.")

client = Groq(api_key=GROQ_API_KEY)

print("Client ready:", MODEL)
print("PLY:", PLY_PATH)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 6.2 MB/s eta 0:00:00
Client ready: openai/gpt-oss-120b
PLY: point_cloud (7).ply


## 2. Spatial memory / persistent identity model

In [2]:
VALID_SESSIONS = ("baseline", "latest")
VALID_REGIONS = ()
SH_COEFF_TO_RGB = 0.28209479177387814


def sigmoid(x):
    x = np.asarray(x, dtype=np.float64)
    return 1.0 / (1.0 + np.exp(-np.clip(x, -80.0, 80.0)))


@dataclass
class Observation:
    session: str
    points: np.ndarray
    colours: np.ndarray
    primitive_ids: np.ndarray
    sh_dc: Optional[np.ndarray] = None
    opacity_logit: Optional[np.ndarray] = None
    scales_log: Optional[np.ndarray] = None
    opacity: Optional[np.ndarray] = None
    scales: Optional[np.ndarray] = None
    rotations: Optional[np.ndarray] = None
    normals: Optional[np.ndarray] = None
    truth_labels: Optional[np.ndarray] = None
    source_file: Optional[str] = None
    source_kind: str = "UNKNOWN"


class SpatialMemory:
    def __init__(self):
        self.observations = {}
        self.trees = {}
        self.identity_index = {}
        self.semantic_prototypes = {}

    def clear_latest(self):
        self.observations.pop("latest", None)
        self.trees.pop("latest", None)
        self.identity_index.pop("latest", None)

    def add_observation(self, observation):
        if observation.session not in VALID_SESSIONS:
            raise ValueError(f"Invalid session: {observation.session}")
        n = len(observation.points)
        if n == 0:
            raise ValueError("observation must contain at least one point")
        if len(observation.colours) != n or len(observation.primitive_ids) != n:
            raise ValueError("points, colours and primitive_ids must have equal length")
        if observation.colours.ndim != 2 or observation.colours.shape[1] != 3:
            raise ValueError("colours must have shape (N, 3)")
        if observation.primitive_ids.ndim != 1:
            raise ValueError("primitive_ids must have shape (N,)")
        if not np.isfinite(observation.points).all():
            raise ValueError("points contain non-finite values")
        if not np.isfinite(observation.colours).all():
            raise ValueError("colours contain non-finite values")
        if observation.sh_dc is not None and len(observation.sh_dc) != n:
            raise ValueError("sh_dc must have the same length as points")
        for name in ("opacity_logit", "scales_log", "opacity", "scales", "rotations", "normals"):
            value = getattr(observation, name)
            if value is not None and len(value) != n:
                raise ValueError(f"{name} must have the same length as points")

        self.observations[observation.session] = observation
        self.trees[observation.session] = KDTree(observation.points)

        index = {}
        for row, gid in enumerate(observation.primitive_ids.astype(np.int64)):
            index.setdefault(int(gid), []).append(row)
        self.identity_index[observation.session] = {
            gid: np.asarray(rows, dtype=np.int32)
            for gid, rows in index.items()
        }

    def require_session(self, session):
        if session not in self.observations:
            raise ValueError(
                f"Unknown/unloaded session '{session}'. Loaded sessions: {list(self.observations)}"
            )
        return self.observations[session]

    def require_latest_pair(self, session_a="baseline", session_b="latest"):
        if session_a not in self.observations or session_b not in self.observations:
            return None
        return self.observations[session_a], self.observations[session_b]

    @staticmethod
    def validate_xyz(x, y, z):
        values = np.asarray([x, y, z], dtype=np.float64)
        if not np.isfinite(values).all():
            raise ValueError("Coordinates must be finite.")
        return values.astype(np.float32)

    @staticmethod
    def validate_radius(radius):
        if not np.isfinite(radius) or radius <= 0 or radius > 2.0:
            raise ValueError("radius must be finite and in (0, 2.0].")
        return float(radius)

    @staticmethod
    def normalized_entropy(counts):
        k = len(counts)
        if k <= 1:
            return 0.0
        p = np.asarray(counts, dtype=np.float64)
        p /= max(p.sum(), 1e-12)
        h = float(-(p * np.log2(p + 1e-12)).sum())
        return h / math.log2(k)

    def probe_point(self, session, xyz, radius=0.15):
        obs = self.require_session(session)
        xyz = self.validate_xyz(*xyz)
        radius = self.validate_radius(radius)
        idx = self.trees[session].query_ball_point(xyz, r=radius)

        if not idx:
            return {
                "ok": True,
                "occupied": False,
                "query_xyz": [round(float(v), 5) for v in xyz],
                "neighbour_count": 0,
                "unique_grain_ids": 0,
                "semantic_status": "UNAVAILABLE",
            }

        ids = obs.primitive_ids[idx].astype(np.int64)
        counts = Counter(int(v) for v in ids)
        dominant_gid, dominant_count = counts.most_common(1)[0]

        result = {
            "ok": True,
            "occupied": True,
            "query_xyz": [round(float(v), 5) for v in xyz],
            "neighbour_count": int(len(idx)),
            "unique_grain_ids": int(len(counts)),
            "grain_consistency_confidence": round(dominant_count / len(idx), 4),
            "normalized_entropy": round(self.normalized_entropy(list(counts.values())), 4),
            "dominant_grain_id": int(dominant_gid),
            "dominant_grain_fraction": round(dominant_count / len(idx), 4),
            "semantic_status": "UNAVAILABLE",
            "mean_colour": [round(float(v), 4) for v in obs.colours[idx].mean(axis=0)],
            "centroid_of_neighbours": [round(float(v), 5) for v in obs.points[idx].mean(axis=0)],
        }
        if obs.opacity is not None:
            result["mean_opacity"] = round(float(obs.opacity[idx].mean()), 5)
            result["opacity_parameterization"] = "sigmoid(opacity_logit)"
        if obs.scales is not None:
            result["mean_scale"] = [round(float(v), 8) for v in obs.scales[idx].mean(axis=0)]
            result["scale_parameterization"] = "exp(scale_log)"
        return result

    def grain_lookup(self, session, grain_id, include_indices=False):
        obs = self.require_session(session)
        try:
            grain_id = int(grain_id)
        except Exception:
            return {"ok": False, "error_type": "INVALID_GRAIN_ID"}

        idx = self.identity_index[session].get(grain_id)
        if idx is None:
            return {
                "ok": False,
                "error_type": "UNKNOWN_GRAIN_ID",
                "grain_id": grain_id,
                "message": "The supplied grain_id does not exist in this session.",
            }

        pts = obs.points[idx]
        result = {
            "ok": True,
            "grain_id": grain_id,
            "session": session,
            "vertex_count": int(len(idx)),
            "centroid": [round(float(v), 6) for v in pts.mean(axis=0)],
            "xyz_bounds_min": [round(float(v), 6) for v in pts.min(axis=0)],
            "xyz_bounds_max": [round(float(v), 6) for v in pts.max(axis=0)],
            "mean_colour": [round(float(v), 5) for v in obs.colours[idx].mean(axis=0)],
            "persistent_identity": "PLY grain_id",
        }
        if obs.opacity is not None:
            result["mean_opacity"] = round(float(obs.opacity[idx].mean()), 6)
            result["opacity_parameterization"] = "sigmoid(opacity_logit)"
        if obs.scales is not None:
            result["mean_scale"] = [round(float(v), 8) for v in obs.scales[idx].mean(axis=0)]
            result["scale_parameterization"] = "exp(scale_log)"
        if obs.normals is not None:
            result["mean_normal"] = [round(float(v), 5) for v in obs.normals[idx].mean(axis=0)]
        if include_indices:
            result["vertex_indices"] = idx.astype(int).tolist()
        return result

    def nearest_grains(self, session, xyz, radius=0.25, top_k=5):
        obs = self.require_session(session)
        xyz = self.validate_xyz(*xyz)
        radius = self.validate_radius(radius)
        if isinstance(top_k, bool) or not isinstance(top_k, (int, np.integer)) or top_k < 1 or top_k > 100:
            raise ValueError("top_k must be an integer in [1, 100].")
        k = min(max(1, top_k * 8), len(obs.points))
        distances, indices = self.trees[session].query(xyz, k=k)
        distances = np.atleast_1d(distances)
        indices = np.atleast_1d(indices)
        grouped = {}
        for d, i in zip(distances, indices):
            if float(d) > radius:
                continue
            gid = int(obs.primitive_ids[int(i)])
            grouped[gid] = min(grouped.get(gid, float("inf")), float(d))
        rows = sorted(grouped.items(), key=lambda kv: kv[1])[:top_k]
        return {
            "ok": True,
            "query_xyz": [round(float(v), 5) for v in xyz],
            "grains": [
                {"grain_id": int(gid), "nearest_distance": round(float(d), 6)}
                for gid, d in rows
            ],
        }

    def scene_summary(self, session):
        obs = self.require_session(session)
        unique_ids, counts = np.unique(obs.primitive_ids.astype(np.int64), return_counts=True)
        return {
            "ok": True,
            "session": session,
            "source_file": obs.source_file,
            "source_kind": obs.source_kind,
            "gaussian_count": int(len(obs.points)),
            "unique_persistent_grain_count": int(len(unique_ids)),
            "grain_id_min": int(unique_ids.min()) if len(unique_ids) else None,
            "grain_id_max": int(unique_ids.max()) if len(unique_ids) else None,
            "mean_gaussians_per_grain": round(float(counts.mean()), 6) if len(counts) else None,
            "median_gaussians_per_grain": float(np.median(counts)) if len(counts) else None,
            "max_gaussians_per_grain": int(counts.max()) if len(counts) else 0,
            "one_gaussian_grain_fraction": round(float(np.mean(counts == 1)), 6) if len(counts) else None,
            "heavy_tail_max_to_median": round(float(counts.max() / max(np.median(counts), 1)), 6) if len(counts) else None,
            "semantic_regions_available": False,
            "semantic_note": "The supplied PLY contains persistent grain IDs but no semantic region labels.",
        }

    def compare_sessions(self, session_a="baseline", session_b="latest"):
        pair = self.require_latest_pair(session_a, session_b)
        if pair is None:
            return {
                "ok": False,
                "error_type": "LATEST_SESSION_UNAVAILABLE",
                "message": "Temporal comparison requires baseline and latest observations.",
                "loaded_sessions": list(self.observations),
            }

        a, b = pair
        ids_a = set(self.identity_index[session_a])
        ids_b = set(self.identity_index[session_b])
        common = ids_a & ids_b
        baseline_retention = len(common) / max(1, len(ids_a))
        latest_overlap = len(common) / max(1, len(ids_b))

        centroid_shifts = []
        scale_changes = []
        opacity_changes = []
        for gid in common:
            ia = self.identity_index[session_a][gid]
            ib = self.identity_index[session_b][gid]
            ca = a.points[ia].mean(axis=0)
            cb = b.points[ib].mean(axis=0)
            centroid_shifts.append(float(np.linalg.norm(ca - cb)))
            if a.scales is not None and b.scales is not None:
                scale_changes.append(float(np.linalg.norm(a.scales[ia].mean(axis=0) - b.scales[ib].mean(axis=0))))
            if a.opacity is not None and b.opacity is not None:
                opacity_changes.append(float(abs(a.opacity[ia].mean() - b.opacity[ib].mean())))

        return {
            "ok": True,
            "session_a": session_a,
            "session_b": session_b,
            "persistent_grains_a": len(ids_a),
            "persistent_grains_b": len(ids_b),
            "common_grains": len(common),
            "baseline_id_retention": round(float(baseline_retention), 6),
            "latest_id_overlap": round(float(latest_overlap), 6),
            "new_grain_ids": int(len(ids_b - ids_a)),
            "lost_grain_ids": int(len(ids_a - ids_b)),
            "median_centroid_shift": round(float(np.median(centroid_shifts)), 6) if centroid_shifts else None,
            "p95_centroid_shift": round(float(np.quantile(centroid_shifts, 0.95)), 6) if centroid_shifts else None,
            "median_scale_change": round(float(np.median(scale_changes)), 6) if scale_changes else None,
            "median_opacity_change": round(float(np.median(opacity_changes)), 6) if opacity_changes else None,
            "verdict": "PERSISTENT-ID DIFFERENCE DETECTED" if (len(ids_a - ids_b) or len(ids_b - ids_a)) else "NO ID-LEVEL LOSS/GAIN DETECTED",
            "registration_note": "This reports ID-level correspondence; it is not a proof of geometric registration quality.",
        }

## 3. Load the supplied LEGO 3DGS PLY with correct parameterization handling

In [4]:
AGENT_CONFIG = {}
if os.path.exists(CONFIG_PATH):
    try:
        AGENT_CONFIG = json.load(open(CONFIG_PATH, "r", encoding="utf-8"))
    except Exception as exc:
        print("Config warning:", exc)


def load_gd3dgr_ply(path, session="baseline", source_kind="ORIGINAL_PLY"):
    from plyfile import PlyData

    ply = PlyData.read(str(path))
    if "vertex" not in ply:
        raise ValueError("PLY does not contain a 'vertex' element.")

    vertex = ply["vertex"].data
    if vertex.dtype.names is None:
        raise ValueError("PLY vertex element has no named properties.")

    names = set(vertex.dtype.names)
    required = ["x", "y", "z", "grain_id", "f_dc_0", "f_dc_1", "f_dc_2"]
    missing = [name for name in required if name not in names]
    if missing:
        raise ValueError("PLY missing required fields: " + ", ".join(missing))

    points = np.column_stack([
        vertex["x"], vertex["y"], vertex["z"]
    ]).astype(np.float32)

    sh_dc = np.column_stack([
        vertex["f_dc_0"],
        vertex["f_dc_1"],
        vertex["f_dc_2"],
    ]).astype(np.float32)

    colours = np.clip(
        0.5 + sh_dc * SH_COEFF_TO_RGB,
        0.0,
        1.0,
    )

    raw_ids = np.asarray(vertex["grain_id"])
    if not np.isfinite(raw_ids).all():
        raise ValueError("PLY grain_id contains non-finite values.")
    if not np.equal(raw_ids, np.floor(raw_ids)).all():
        raise ValueError("PLY grain_id must contain integer-valued IDs.")
    primitive_ids = raw_ids.astype(np.int64)

    opacity_logit = (
        np.asarray(vertex["opacity"], dtype=np.float32)
        if "opacity" in names
        else None
    )
    opacity = sigmoid(opacity_logit).astype(np.float32) if opacity_logit is not None else None

    scales_log = (
        np.column_stack([
            vertex[f"scale_{i}"] for i in range(3)
        ]).astype(np.float32)
        if all(f"scale_{i}" in names for i in range(3))
        else None
    )
    scales = np.exp(np.clip(scales_log, -80, 80)).astype(np.float32) if scales_log is not None else None

    rotations = (
        np.column_stack([
            vertex[f"rot_{i}"] for i in range(4)
        ]).astype(np.float32)
        if all(f"rot_{i}" in names for i in range(4))
        else None
    )

    normals = (
        np.column_stack([
            vertex[f"n{axis}"] for axis in ("x", "y", "z")
        ]).astype(np.float32)
        if all(f"n{axis}" in names for axis in ("x", "y", "z"))
        else None
    )

    if len(np.unique(primitive_ids)) == len(vertex):
        raise ValueError(
            "The PLY grain_id field is not grouping vertices; "
            "persistent grain identity would be ineffective."
        )

    return Observation(
        session=session,
        points=points,
        colours=colours,
        primitive_ids=primitive_ids,
        sh_dc=sh_dc,
        opacity_logit=opacity_logit,
        scales_log=scales_log,
        opacity=opacity,
        scales=scales,
        rotations=rotations,
        normals=normals,
        source_file=str(path),
        source_kind=source_kind,
    )


BASELINE = load_gd3dgr_ply(
    PLY_PATH,
    "baseline",
    "ORIGINAL_PLY",
)
LATEST = None

memory = SpatialMemory()
memory.add_observation(BASELINE)

print("Loaded LEGO PLY successfully.")
print(f"Gaussians / vertices: {len(BASELINE.points):,}")
print(f"Persistent grains:    {len(np.unique(BASELINE.primitive_ids)):,}")
print(
    f"grain_id range:       {int(BASELINE.primitive_ids.min())} .. "
    f"{int(BASELINE.primitive_ids.max())}"
)
print(f"XYZ min:              {BASELINE.points.min(axis=0)}")
print(f"XYZ max:              {BASELINE.points.max(axis=0)}")

Loaded LEGO PLY successfully.
Gaussians / vertices: 194,018
Persistent grains:    35,731
grain_id range:       2 .. 40519
XYZ min:              [-0.72575283 -1.3045346  -0.4333119 ]
XYZ max:              [0.7511529 1.2701358 1.0864244]


## 4. Schema / identity verification

In [5]:
summary0 = memory.scene_summary("baseline")
assert summary0["gaussian_count"] == len(BASELINE.points)
assert summary0["unique_persistent_grain_count"] == len(np.unique(BASELINE.primitive_ids))
assert np.isfinite(BASELINE.points).all()
assert np.isfinite(BASELINE.colours).all()
if BASELINE.opacity is not None:
    assert np.logical_and(BASELINE.opacity >= 0.0, BASELINE.opacity <= 1.0).all()
if BASELINE.scales is not None:
    assert np.isfinite(BASELINE.scales).all()
    assert (BASELINE.scales > 0).all()

mid = BASELINE.points[len(BASELINE.points) // 2]
probe_check = memory.probe_point(
    "baseline",
    mid,
    radius=0.02,
)
lookup_check = memory.grain_lookup(
    "baseline",
    int(probe_check["dominant_grain_id"]),
)

print("Schema / identity verification passed.")
print(
    "Example decoded opacity:",
    probe_check.get("mean_opacity"),
    probe_check.get("opacity_parameterization"),
)
print(
    "Example decoded scale:",
    probe_check.get("mean_scale"),
    probe_check.get("scale_parameterization"),
)

Schema / identity verification passed.
Example decoded opacity: 0.23553 sigmoid(opacity_logit)
Example decoded scale: [0.00310207, 0.00381384, 0.00103646] exp(scale_log)


## 5. Region discovery — hypotheses only

In [6]:
class LEGORegionAnalyzer:
    def __init__(self, memory):
        self.memory = memory
        self.cache = {}

    def grain_table(self, session="baseline"):
        obs = self.memory.require_session(session)
        rows = []
        for gid, idx in self.memory.identity_index[session].items():
            p = obs.points[idx]
            c = obs.colours[idx]
            ext = p.max(axis=0) - p.min(axis=0)
            rows.append({
                "grain_id": int(gid),
                "vertex_count": int(len(idx)),
                "cx": float(p[:, 0].mean()),
                "cy": float(p[:, 1].mean()),
                "cz": float(p[:, 2].mean()),
                "ex": float(ext[0]),
                "ey": float(ext[1]),
                "ez": float(ext[2]),
                "r": float(c[:, 0].mean()),
                "g": float(c[:, 1].mean()),
                "b": float(c[:, 2].mean()),
                "brightness": float(c.mean()),
            })
        return pd.DataFrame(rows)

    @staticmethod
    def scene_scale(points):
        return float(np.linalg.norm(points.max(axis=0) - points.min(axis=0)))

    @staticmethod
    def cluster_geometry(points):
        if len(points) < 3:
            return {
                "circularity": 0.0,
                "planarity": 0.0,
                "thickness_ratio": 1.0,
                "principal_axes": np.eye(3).tolist(),
                "eigenvalues": [0, 0, 0],
            }
        x = points - points.mean(axis=0, keepdims=True)
        cov = np.cov(x.T)
        vals, axes = np.linalg.eigh(cov)
        order = np.argsort(vals)[::-1]
        vals = np.maximum(vals[order], 1e-12)
        axes = axes[:, order]
        circularity = float(1.0 - abs(vals[0] - vals[1]) / max(vals[0], 1e-12))
        thickness_ratio = float(np.sqrt(vals[2]) / max(np.sqrt(vals[0]), 1e-12))
        planarity = float(max(0.0, 1.0 - thickness_ratio))
        return {
            "circularity": round(max(0.0, min(1.0, circularity)), 4),
            "planarity": round(planarity, 4),
            "thickness_ratio": round(thickness_ratio, 4),
            "principal_axes": axes.tolist(),
            "eigenvalues": vals.tolist(),
        }

    def detect_regions(
        self,
        session="baseline",
        dark_threshold=0.30,
        eps=None,
        min_samples=7,
    ):
        cache_key = (
            session,
            float(dark_threshold),
            None if eps is None else float(eps),
            int(min_samples),
        )
        if cache_key in self.cache:
            return self.cache[cache_key]

        df = self.grain_table(session)
        obs = self.memory.require_session(session)
        if df.empty:
            return {
                "ok": False,
                "error_type": "EMPTY_SCENE",
                "regions": [],
            }

        scene_scale = self.scene_scale(obs.points)
        if eps is None:
            eps = float(np.clip(0.025 * scene_scale, 0.015, 0.25))

        dark_df = df.loc[df["brightness"].to_numpy() <= float(dark_threshold)].copy()
        if len(dark_df) < min_samples:
            dark_df = df.copy()

        if len(dark_df) < min_samples:
            result = {
                "ok": False,
                "error_type": "INSUFFICIENT_DATA",
                "message": f"Scene contains {len(dark_df)} grains, less than min_samples={min_samples}.",
                "regions": [],
            }
            self.cache[cache_key] = result
            return result

        labels = DBSCAN(
            eps=float(eps),
            min_samples=int(min_samples),
            n_jobs=-1,
        ).fit_predict(
            dark_df[["cx", "cy", "cz"]].to_numpy(np.float32)
        )

        regions = []
        for cluster_id in sorted(set(int(x) for x in labels if int(x) >= 0)):
            sub = dark_df.loc[labels == cluster_id]
            if len(sub) < min_samples:
                continue
            grain_ids = sub["grain_id"].astype(int).tolist()
            pts = np.concatenate([
                obs.points[self.memory.identity_index[session][gid]]
                for gid in grain_ids
            ])
            geom = self.cluster_geometry(pts)
            centroid = pts.mean(axis=0)
            ext = pts.max(axis=0) - pts.min(axis=0)
            dark_fraction = float((sub["brightness"] <= dark_threshold).mean())
            size_score = float(np.clip(len(grain_ids) / 250.0, 0.0, 1.0))
            wheel_score = (
                0.45 * geom["circularity"]
                + 0.25 * geom["planarity"]
                + 0.20 * dark_fraction
                + 0.10 * size_score
            )
            fraction_of_scene = len(grain_ids) / max(1, len(df))
            oversized = fraction_of_scene >= 0.25
            hypothesis = (
                "wheel"
                if (
                    not oversized
                    and wheel_score >= 0.58
                    and geom["circularity"] >= 0.50
                    and geom["thickness_ratio"] <= 0.55
                )
                else "geometric_cluster"
            )
            region_name = f"wheel_candidate_{cluster_id}" if hypothesis == "wheel" else f"region_candidate_{cluster_id}"
            regions.append({
                "region_id": f"candidate_{cluster_id}",
                "region_name": region_name,
                "hypothesis": hypothesis,
                "confidence": round(float(np.clip(wheel_score, 0.0, 1.0)), 4),
                "grain_count": int(len(grain_ids)),
                "scene_grain_fraction": round(float(fraction_of_scene), 6),
                "quality_flag": "OVERSIZED_UNCALIBRATED_CLUSTER" if oversized else "NORMAL_CLUSTER_SIZE",
                "grain_ids": grain_ids,
                "centroid": [round(float(v), 6) for v in centroid],
                "bounds_min": [round(float(v), 6) for v in pts.min(axis=0)],
                "bounds_max": [round(float(v), 6) for v in pts.max(axis=0)],
                "extent": [round(float(v), 6) for v in ext],
                "dark_fraction": round(dark_fraction, 4),
                "geometry": geom,
                "inspection_note": "Geometry/appearance hypothesis only; no ground-truth semantic label is present in the PLY.",
            })

        regions.sort(key=lambda r: r["confidence"], reverse=True)
        for rank, region in enumerate(regions, start=1):
            region["rank"] = rank

        result = {
            "ok": True,
            "session": session,
            "method": "dark-appearance clustering + 3-D PCA wheel-likeness",
            "calibration_status": "UNCALIBRATED_HEURISTIC",
            "scene_scale": round(scene_scale, 6),
            "dbscan_eps": round(float(eps), 6),
            "dark_threshold": float(dark_threshold),
            "min_samples": int(min_samples),
            "region_count": int(len(regions)),
            "wheel_candidate_count": int(sum(r["hypothesis"] == "wheel" for r in regions)),
            "regions": regions,
            "semantic_warning": "These are geometric hypotheses, not ground-truth semantic labels.",
        }
        self.cache[cache_key] = result
        return result

    def region_lookup(self, session, region_name):
        result = self.detect_regions(session)
        if not result.get("ok"):
            return result
        for region in result.get("regions", []):
            if region["region_name"] == region_name or region["region_id"] == region_name:
                return {"ok": True, **region}
        return {
            "ok": False,
            "error_type": "UNKNOWN_REGION",
            "region_name": region_name,
            "message": "region_name must exactly match a name previously returned by detect_regions.",
        }


REGION_ANALYZER = LEGORegionAnalyzer(memory)
REGION_HYPOTHESES = REGION_ANALYZER.detect_regions("baseline")
print(
    "Region hypotheses:",
    REGION_HYPOTHESES.get("region_count", 0),
)
print(
    "Wheel candidates:",
    REGION_HYPOTHESES.get("wheel_candidate_count", 0),
)

Region hypotheses: 5
Wheel candidates: 1


## 6. Temporal / damage analyzer + synthetic PLY writer

In [7]:
class LEGODamageAnalyzer:
    def __init__(self, memory, region_analyzer):
        self.memory = memory
        self.region_analyzer = region_analyzer

    def load_latest(
        self,
        path,
        min_baseline_id_retention=0.90,
        min_latest_id_overlap=0.90,
        scale_ratio_bounds=(0.97, 1.03),
    ):
        obs = load_gd3dgr_ply(
            path,
            "latest",
            source_kind="REGISTERED_LATEST_PLY",
        )
        baseline = self.memory.require_session("baseline")
        ids_a = set(self.memory.identity_index["baseline"])
        ids_b = set(np.unique(obs.primitive_ids).astype(np.int64))
        common = ids_a & ids_b
        baseline_retention = len(common) / max(1, len(ids_a))
        latest_overlap = len(common) / max(1, len(ids_b))

        if baseline_retention < min_baseline_id_retention or latest_overlap < min_latest_id_overlap:
            raise ValueError(
                "Latest PLY rejected: strict persistent-ID compatibility failed "
                f"(baseline retention={baseline_retention:.4f}, latest overlap={latest_overlap:.4f}); "
                f"required >= {min_baseline_id_retention:.2f} and >= {min_latest_id_overlap:.2f}."
            )

        a_extent = baseline.points.max(axis=0) - baseline.points.min(axis=0)
        b_extent = obs.points.max(axis=0) - obs.points.min(axis=0)
        scale_ratio = float(
            np.linalg.norm(b_extent) / max(np.linalg.norm(a_extent), 1e-12)
        )
        low, high = scale_ratio_bounds
        if not low <= scale_ratio <= high:
            raise ValueError(
                "Latest PLY rejected: scene-scale ratio is outside the strict registered range "
                f"[{low:.3f}, {high:.3f}] (observed {scale_ratio:.4f})."
            )

        self.memory.clear_latest()
        self.memory.add_observation(obs)
        return {
            "ok": True,
            "session": "latest",
            "source_file": str(path),
            "gaussian_count": int(len(obs.points)),
            "unique_grain_count": int(len(ids_b)),
            "persistent_id_overlap": round(float(latest_overlap), 5),
            "scene_scale_ratio": round(scale_ratio, 5),
        }

    def detect_damage(
        self,
        baseline_session="baseline",
        latest_session="latest",
        partial_retention_threshold=0.75,
        min_partial_gaussians=2,
        min_samples=5,
        eps=None,
    ):
        pair = self.memory.require_latest_pair(baseline_session, latest_session)
        if pair is None:
            return {
                "ok": False,
                "error_type": "LATEST_SESSION_UNAVAILABLE",
                "message": "Damage detection requires baseline and latest observations.",
            }
        if not 0.0 < partial_retention_threshold < 1.0:
            return {
                "ok": False,
                "error_type": "INVALID_PARTIAL_RETENTION_THRESHOLD",
            }

        base_idx = self.memory.identity_index[baseline_session]
        latest_idx = self.memory.identity_index[latest_session]
        missing_ids = []
        partial_ids = []
        affected_partial = []

        for gid, ia in base_idx.items():
            base_count = len(ia)
            latest_count = len(latest_idx.get(gid, []))
            if latest_count == 0:
                missing_ids.append(int(gid))
            elif base_count >= min_partial_gaussians:
                retention = latest_count / max(base_count, 1)
                if retention < partial_retention_threshold:
                    partial_ids.append(int(gid))
                    affected_partial.append({
                        "grain_id": int(gid),
                        "baseline_gaussian_count": int(base_count),
                        "latest_gaussian_count": int(latest_count),
                        "retained_fraction": round(float(retention), 6),
                        "damage_signal": "WITHIN_GRAIN_COUNT_LOSS",
                    })

        base_obs = self.memory.require_session(baseline_session)
        if missing_ids:
            missing_centroids = np.asarray([
                base_obs.points[base_idx[gid]].mean(axis=0)
                for gid in missing_ids
            ], dtype=np.float32)
        else:
            missing_centroids = np.empty((0, 3), dtype=np.float32)

        if len(missing_centroids) == 0:
            zones = []
        elif len(missing_centroids) < min_samples:
            zones = [{
                "cluster_id": -1,
                "grain_count": int(len(missing_centroids)),
                "severity": "MODERATE",
                "center": [round(float(v), 6) for v in missing_centroids.mean(axis=0)],
            }]
        else:
            if eps is None:
                scene_scale = self.region_analyzer.scene_scale(base_obs.points)
                eps = float(np.clip(0.03 * scene_scale, 0.015, 0.25))
            labels = DBSCAN(
                eps=float(eps),
                min_samples=int(min_samples),
                n_jobs=-1,
            ).fit_predict(missing_centroids)
            zones = []
            for cid in sorted(set(int(x) for x in labels if int(x) >= 0)):
                idx = np.where(labels == cid)[0]
                pts = missing_centroids[idx]
                zones.append({
                    "cluster_id": int(cid),
                    "grain_count": int(len(idx)),
                    "severity": "CRITICAL" if len(idx) > 50 else "MODERATE",
                    "center": {
                        "x": round(float(pts[:, 0].mean()), 4),
                        "y": round(float(pts[:, 1].mean()), 4),
                        "z": round(float(pts[:, 2].mean()), 4),
                    },
                })
            noise_idx = np.where(labels == -1)[0]
            if len(noise_idx):
                pts = missing_centroids[noise_idx]
                zones.append({
                    "cluster_id": -1,
                    "grain_count": int(len(noise_idx)),
                    "severity": "MODERATE",
                    "center": {
                        "x": round(float(pts[:, 0].mean()), 4),
                        "y": round(float(pts[:, 1].mean()), 4),
                        "z": round(float(pts[:, 2].mean()), 4),
                    },
                })

        zones.sort(key=lambda z: z["grain_count"], reverse=True)
        affected_partial.sort(key=lambda x: x["grain_id"])

        return {
            "ok": True,
            "damage_detected": bool(missing_ids or partial_ids),
            "total_missing_grains": int(len(missing_ids)),
            "missing_grains": int(len(missing_ids)),
            "partially_reduced_grains": int(len(partial_ids)),
            "missing_grain_ids": missing_ids[:100],
            "partial_grain_ids": partial_ids[:100],
            "damage_zones_count": int(len(zones)),
            "damage_zones": zones,
            "affected_grain_count": int(len(missing_ids) + len(partial_ids)),
            "partial_grain_loss": affected_partial[:100],
            "granularity_warning": (
                "Persistent grain IDs group a heavy-tailed number of Gaussians. "
                "Missing IDs are a binary disappearance signal; within-grain count loss "
                "is a separate, lower-confidence signal derived from retained Gaussian counts."
            ),
            "partial_retention_threshold": float(partial_retention_threshold),
            "min_partial_gaussians": int(min_partial_gaussians),
        }

    @staticmethod
    def _height_grid(points, height_axis=2, xy_voxel=0.02, percentile=90.0):
        plane_axes = [a for a in range(3) if a != height_axis]
        coords = points[:, plane_axes]
        keys = np.floor(coords / float(xy_voxel)).astype(np.int64)
        buckets = {}
        for i, key in enumerate(keys):
            buckets.setdefault((int(key[0]), int(key[1])), []).append(i)
        result = {}
        for key, idx in buckets.items():
            result[key] = {
                "height": float(np.percentile(points[idx, height_axis], percentile)),
                "count": int(len(idx)),
            }
        return result, plane_axes

    def delta_h(
        self,
        session_a="baseline",
        session_b="latest",
        xy_voxel=0.02,
        height_axis=2,
        height_percentile=90.0,
        min_cell_points=3,
        top_k=25,
    ):
        if not np.isfinite(xy_voxel) or xy_voxel <= 0:
            return {"ok": False, "error_type": "INVALID_XY_VOXEL"}
        if height_axis not in (0, 1, 2):
            return {"ok": False, "error_type": "INVALID_HEIGHT_AXIS"}
        if not np.isfinite(height_percentile) or not 0 < height_percentile <= 100:
            return {"ok": False, "error_type": "INVALID_HEIGHT_PERCENTILE"}
        if not isinstance(top_k, (int, np.integer)) or top_k < 1 or top_k > 100:
            return {"ok": False, "error_type": "INVALID_TOP_K"}
        if not isinstance(min_cell_points, (int, np.integer)) or min_cell_points < 1:
            return {"ok": False, "error_type": "INVALID_MIN_CELL_POINTS"}

        pair = self.memory.require_latest_pair(session_a, session_b)
        if pair is None:
            return {
                "ok": False,
                "error_type": "LATEST_SESSION_UNAVAILABLE",
                "message": "ΔH requires a second registered PLY loaded as latest.",
            }

        a, b = pair
        ga, axes = self._height_grid(a.points, height_axis, xy_voxel, height_percentile)
        gb, _ = self._height_grid(b.points, height_axis, xy_voxel, height_percentile)
        common = sorted(set(ga) & set(gb))
        deltas = []
        for key in common:
            if ga[key]["count"] < min_cell_points or gb[key]["count"] < min_cell_points:
                continue
            deltas.append((key, float(gb[key]["height"] - ga[key]["height"])))
        if not deltas:
            return {
                "ok": False,
                "error_type": "NO_COMMON_HEIGHT_CELLS",
                "message": "No sufficiently populated common XY cells were found.",
            }

        vals = np.asarray([v for _, v in deltas], dtype=np.float64)
        median = float(np.median(vals))
        mad = float(np.median(np.abs(vals - median)))
        scene_extent = a.points.max(axis=0) - a.points.min(axis=0)
        scene_scale = float(np.linalg.norm(scene_extent))
        noise_floor = max(1e-6, 1.4826 * mad)
        threshold = max(3.0 * noise_floor, 0.002 * scene_scale, 2.0 * xy_voxel)
        damage_cells = [(k, v) for k, v in deltas if v <= median - threshold]
        damage_cells.sort(key=lambda kv: kv[1])

        return {
            "ok": True,
            "method": "registered XY height-grid change (latest - baseline)",
            "height_axis": int(height_axis),
            "height_percentile": float(height_percentile),
            "xy_voxel": float(xy_voxel),
            "median_delta_h": round(median, 6),
            "mad_delta_h": round(mad, 6),
            "damage_threshold": round(float(threshold), 6),
            "common_cells": int(len(deltas)),
            "flagged_damage_cells": int(len(damage_cells)),
            "median_absolute_delta_h": round(float(np.median(np.abs(vals))), 6),
            "max_negative_delta_h": round(float(damage_cells[0][1]), 6) if damage_cells else None,
            "top_damage_cells": [
                {
                    "grid_key": [int(k[0]), int(k[1])],
                    "delta_h": round(float(v), 6),
                    "baseline_height": round(float(ga[k]["height"]), 6),
                    "latest_height": round(float(gb[k]["height"]), 6),
                }
                for k, v in damage_cells[:top_k]
            ],
            "interpretation": (
                "Negative ΔH means the latest scan is lower than baseline at that registered XY location; "
                "it is damage evidence only when registration and height-axis calibration are valid."
            ),
        }

    def simulate_damage(
        self,
        region_name="wheel_candidate_1",
        depth=0.03,
        radius_scale=0.75,
        height_axis=2,
    ):
        if not np.isfinite(depth) or depth <= 0:
            return {"ok": False, "error_type": "INVALID_DAMAGE_DEPTH", "message": "depth must be > 0."}
        if not np.isfinite(radius_scale) or radius_scale <= 0 or radius_scale > 5:
            return {"ok": False, "error_type": "INVALID_RADIUS_SCALE", "message": "radius_scale must be in (0, 5]."}
        if height_axis not in (0, 1, 2):
            return {"ok": False, "error_type": "INVALID_HEIGHT_AXIS", "message": "height_axis must be 0, 1, or 2."}

        lookup = self.region_analyzer.region_lookup("baseline", region_name)
        if not lookup.get("ok"):
            return lookup
        obs = self.memory.require_session("baseline")
        center = np.asarray(lookup["centroid"], dtype=np.float32)
        ext = np.asarray(lookup["extent"], dtype=np.float32)
        radius = float(max(radius_scale * float(np.max(ext)) * 0.35, depth * 1.5))
        distances = np.linalg.norm(obs.points - center, axis=1)
        height = obs.points[:, height_axis]
        cut_floor = float(center[height_axis] - depth)
        damaged = (distances <= radius) & (height >= cut_floor)
        keep = ~damaged
        if int(np.sum(keep)) < int(0.5 * len(keep)):
            return {"ok": False, "error_type": "DAMAGE_MASK_TOO_LARGE", "removed_fraction": float(damaged.mean())}

        synthetic = Observation(
            session="latest",
            points=obs.points[keep],
            colours=obs.colours[keep],
            primitive_ids=obs.primitive_ids[keep],
            sh_dc=obs.sh_dc[keep] if obs.sh_dc is not None else None,
            opacity_logit=obs.opacity_logit[keep] if obs.opacity_logit is not None else None,
            scales_log=obs.scales_log[keep] if obs.scales_log is not None else None,
            opacity=obs.opacity[keep] if obs.opacity is not None else None,
            scales=obs.scales[keep] if obs.scales is not None else None,
            rotations=obs.rotations[keep] if obs.rotations is not None else None,
            normals=obs.normals[keep] if obs.normals is not None else None,
            source_file=f"synthetic_damage::{region_name}",
            source_kind="SYNTHETIC_DAMAGE",
        )
        self.memory.clear_latest()
        self.memory.add_observation(synthetic)
        return {
            "ok": True,
            "mode": "SYNTHETIC_DAMAGE_TEST",
            "region_name": region_name,
            "removed_gaussians": int(damaged.sum()),
            "removed_fraction": round(float(damaged.mean()), 6),
            "cut_floor": round(cut_floor, 6),
            "radius": round(radius, 6),
            "warning": "This is a controlled perturbation for validating damage detection.",
        }


DAMAGE_ANALYZER = LEGODamageAnalyzer(
    memory,
    REGION_ANALYZER,
)
print("Temporal/damage analyzer ready.")

Temporal/damage analyzer ready.


## 7. Synthetic damage PLY generation — automatic validation copy of the supplied PLY

In [8]:
def save_gaussian_observation_to_ply(
    filepath,
    observation,
    keep_mask=None,
):
    """
    Save an Observation back to a binary little-endian PLY.

    Important:
    - opacity is stored back as the RAW 3DGS opacity_logit
    - scales are stored back as RAW log-scales
    - colours are converted back to 3DGS DC coefficients
    - persistent grain_id is preserved exactly
    """

    from plyfile import PlyData, PlyElement

    if not isinstance(
        observation,
        Observation,
    ):
        raise TypeError(
            "observation must be an Observation instance."
        )

    n = len(observation.points)

    if keep_mask is None:
        keep_mask = np.ones(
            n,
            dtype=bool,
        )
    else:
        keep_mask = np.asarray(
            keep_mask,
            dtype=bool,
        )

        if keep_mask.ndim != 1:
            raise ValueError(
                "keep_mask must be a 1-D boolean array."
            )

        if len(keep_mask) != n:
            raise ValueError(
                "keep_mask must have the same length as observation."
            )

    if not keep_mask.any():
        raise ValueError(
            "keep_mask removes every Gaussian."
        )

    # --------------------------------------------------------
    # Select retained rows
    # --------------------------------------------------------
    points = observation.points[
        keep_mask
    ]

    colours = observation.colours[
        keep_mask
    ]

    primitive_ids = observation.primitive_ids[
        keep_mask
    ]

    # --------------------------------------------------------
    # Recover RAW 3DGS opacity logits
    #
    # Observation.opacity is decoded sigmoid(opacity_logit).
    # Convert back using the numerically stable logit.
    # --------------------------------------------------------
    opacity_logit = None

    if observation.opacity is not None:
        decoded_opacity = np.asarray(
            observation.opacity[
                keep_mask
            ],
            dtype=np.float32,
        )

        decoded_opacity = np.clip(
            decoded_opacity,
            1e-6,
            1.0 - 1e-6,
        )

        opacity_logit = (
            np.log(decoded_opacity)
            - np.log1p(-decoded_opacity)
        ).astype(
            np.float32
        )

    # --------------------------------------------------------
    # Recover RAW 3DGS log-scales
    #
    # Observation.scales is decoded exp(scale_log).
    # --------------------------------------------------------
    scales_log = None

    if observation.scales is not None:
        decoded_scales = np.asarray(
            observation.scales[
                keep_mask
            ],
            dtype=np.float32,
        )

        if not np.isfinite(
            decoded_scales
        ).all():
            raise ValueError(
                "Decoded scales contain non-finite values."
            )

        if np.any(
            decoded_scales <= 0
        ):
            raise ValueError(
                "Decoded 3DGS scales must be > 0 "
                "before converting back to log-scales."
            )

        scales_log = np.log(
            decoded_scales
        ).astype(
            np.float32
        )

    dc = (
        (
            colours
            - 0.5
        )
        / SH_COEFF_TO_RGB
    ).astype(
        np.float32
    )

    # --------------------------------------------------------
    # Optional arrays
    # --------------------------------------------------------
    rotations = (
        observation.rotations[
            keep_mask
        ].astype(
            np.float32
        )
        if observation.rotations is not None
        else None
    )

    normals = (
        observation.normals[
            keep_mask
        ].astype(
            np.float32
        )
        if observation.normals is not None
        else None
    )

    # --------------------------------------------------------
    # Build structured PLY dtype FIRST
    # This avoids the previous uninitialized `header` bug.
    # --------------------------------------------------------
    properties = [
        ("x", "f4"),
        ("y", "f4"),
        ("z", "f4"),
        ("f_dc_0", "f4"),
        ("f_dc_1", "f4"),
        ("f_dc_2", "f4"),
        ("grain_id", "i4"),
    ]

    if opacity_logit is not None:
        properties.append(
            ("opacity", "f4")
        )

    if scales_log is not None:
        properties.extend(
            [
                ("scale_0", "f4"),
                ("scale_1", "f4"),
                ("scale_2", "f4"),
            ]
        )

    if rotations is not None:
        properties.extend(
            [
                ("rot_0", "f4"),
                ("rot_1", "f4"),
                ("rot_2", "f4"),
                ("rot_3", "f4"),
            ]
        )

    if normals is not None:
        properties.extend(
            [
                ("nx", "f4"),
                ("ny", "f4"),
                ("nz", "f4"),
            ]
        )

    vertex_data = np.empty(
        len(points),
        dtype=np.dtype(
            properties
        ),
    )

    # --------------------------------------------------------
    # Populate mandatory fields
    # --------------------------------------------------------
    vertex_data["x"] = (
        points[:, 0]
    )
    vertex_data["y"] = (
        points[:, 1]
    )
    vertex_data["z"] = (
        points[:, 2]
    )

    vertex_data["f_dc_0"] = (
        dc[:, 0]
    )
    vertex_data["f_dc_1"] = (
        dc[:, 1]
    )
    vertex_data["f_dc_2"] = (
        dc[:, 2]
    )

    # Preserve persistent identity exactly.
    if (
        primitive_ids.min()
        < np.iinfo(
            np.int32
        ).min
        or primitive_ids.max()
        > np.iinfo(
            np.int32
        ).max
    ):
        raise ValueError(
            "grain_id values exceed int32 range."
        )

    vertex_data["grain_id"] = (
        primitive_ids.astype(
            np.int32
        )
    )

    # --------------------------------------------------------
    # Populate optional fields
    # --------------------------------------------------------
    if opacity_logit is not None:
        vertex_data["opacity"] = (
            opacity_logit
        )

    if scales_log is not None:
        vertex_data["scale_0"] = (
            scales_log[:, 0]
        )
        vertex_data["scale_1"] = (
            scales_log[:, 1]
        )
        vertex_data["scale_2"] = (
            scales_log[:, 2]
        )

    if rotations is not None:
        vertex_data["rot_0"] = (
            rotations[:, 0]
        )
        vertex_data["rot_1"] = (
            rotations[:, 1]
        )
        vertex_data["rot_2"] = (
            rotations[:, 2]
        )
        vertex_data["rot_3"] = (
            rotations[:, 3]
        )

    if normals is not None:
        vertex_data["nx"] = (
            normals[:, 0]
        )
        vertex_data["ny"] = (
            normals[:, 1]
        )
        vertex_data["nz"] = (
            normals[:, 2]
        )


    vertex_element = PlyElement.describe(
        vertex_data,
        "vertex",
    )

    PlyData(
        [vertex_element],
        text=False,
        byte_order="<",
    ).write(
        str(filepath)
    )


def create_synthetic_damaged_ply(
    baseline_obs,
    output_path="latest_damaged_scene.ply",
    damage_center=None,
    radius=0.35,
    height_axis=2,
    depth_fraction=0.90,
):
    """
    Create a controlled synthetic-damage PLY from the supplied
    baseline PLY.

    The baseline itself is never modified.

    Default parameters reproduce the original experiment:
        radius=0.35
        height_axis=2
        depth_fraction=0.90
    """

    if not isinstance(
        baseline_obs,
        Observation,
    ):
        raise TypeError(
            "baseline_obs must be an Observation."
        )

    if (
        not np.isfinite(radius)
        or radius <= 0
    ):
        raise ValueError(
            "radius must be finite and > 0."
        )

    if height_axis not in (
        0,
        1,
        2,
    ):
        raise ValueError(
            "height_axis must be 0, 1, or 2."
        )

    if (
        not np.isfinite(
            depth_fraction
        )
        or not 0 < depth_fraction <= 1
    ):
        raise ValueError(
            "depth_fraction must be in (0, 1]."
        )

    pts = baseline_obs.points

    # Original behavior: use the scene median when no explicit
    # damage center is supplied.
    if damage_center is None:
        damage_center = np.median(
            pts,
            axis=0,
        )

    damage_center = np.asarray(
        damage_center,
        dtype=np.float32,
    )

    if damage_center.shape != (
        3,
    ):
        raise ValueError(
            "damage_center must contain exactly 3 coordinates."
        )

    distances = np.linalg.norm(
        pts
        - damage_center,
        axis=1,
    )

    target_indices = np.where(
        distances <= float(radius)
    )[0]

    if len(target_indices) == 0:
        raise ValueError(
            "No Gaussians found within the requested "
            f"damage radius {radius} of "
            f"damage center {damage_center.tolist()}."
        )

    # Slice from the top downward along the selected height axis.
    sorted_target_indices = (
        target_indices[
            np.argsort(
                -pts[
                    target_indices,
                    height_axis,
                ]
            )
        ]
    )

    num_to_remove = int(
        len(
            sorted_target_indices
        )
        * depth_fraction
    )

    # Guarantee a meaningful synthetic perturbation.
    num_to_remove = min(
        max(
            num_to_remove,
            1,
        ),
        len(
            sorted_target_indices
        ),
    )

    indices_to_remove = (
        sorted_target_indices[
            :num_to_remove
        ]
    )

    keep_mask = np.ones(
        len(pts),
        dtype=bool,
    )

    keep_mask[
        indices_to_remove
    ] = False

    save_gaussian_observation_to_ply(
        output_path,
        baseline_obs,
        keep_mask,
    )

    return {
        "ok": True,
        "output_file": str(
            output_path
        ),
        "original_gaussian_count": int(
            len(pts)
        ),
        "damaged_gaussian_count": int(
            np.sum(
                keep_mask
            )
        ),
        "removed_gaussians": int(
            num_to_remove
        ),
        "damage_center": [
            float(v)
            for v in damage_center
        ],
        "damage_radius": float(
            radius
        ),
    }


# ============================================================
# AUTOMATIC SYNTHETIC VALIDATION ARTIFACT
# ============================================================

SYNTHETIC_PLY_PATH = (
    "latest_damaged_scene.ply"
)

synthetic_result = (
    create_synthetic_damaged_ply(
        BASELINE,
        SYNTHETIC_PLY_PATH,
        radius=0.35,
        height_axis=2,
        depth_fraction=0.90,
    )
)

print(
    "Synthetic PLY Created:",
    synthetic_result,
)

# Load the generated damaged PLY as the latest session.
load_result = DAMAGE_ANALYZER.load_latest(
    SYNTHETIC_PLY_PATH
)

print(
    "Loaded Session Status:",
    load_result,
)

Synthetic PLY Created: {'ok': True, 'output_file': 'latest_damaged_scene.ply', 'original_gaussian_count': 194018, 'damaged_gaussian_count': 182928, 'removed_gaussians': 11090, 'damage_center': [-0.013445455580949783, -0.09575404226779938, 0.08590379357337952], 'damage_radius': 0.35}
Loaded Session Status: {'ok': True, 'session': 'latest', 'source_file': 'latest_damaged_scene.ply', 'gaussian_count': 182928, 'unique_grain_count': 33614, 'persistent_id_overlap': 1.0, 'scene_scale_ratio': 1.0}


## 8. Tool schemas + dispatch

In [9]:
TOOLS = [
    {"type":"function","function":{
        "name":"probe_point",
        "description":"Probe a LEGO coordinate and return occupancy and persistent grain identity evidence.",
        "parameters":{"type":"object","properties":{
            "session":{"type":"string","enum":["baseline","latest"]},
            "x":{"type":"number"},"y":{"type":"number"},"z":{"type":"number"},
            "radius":{"type":"number"}
        },"required":["session","x","y","z"]}
    }},
    {"type":"function","function":{
        "name":"grain_lookup",
        "description":"Resolve an exact persistent grain_id to verified geometry and decoded appearance statistics.",
        "parameters":{"type":"object","properties":{
            "session":{"type":"string","enum":["baseline","latest"]},
            "grain_id":{"type":"integer"},
            "include_indices":{"type":"boolean"}
        },"required":["session","grain_id"]}
    }},
    {"type":"function","function":{
        "name":"nearest_grains",
        "description":"Return persistent grain IDs nearest to a coordinate.",
        "parameters":{"type":"object","properties":{
            "session":{"type":"string","enum":["baseline","latest"]},
            "x":{"type":"number"},"y":{"type":"number"},"z":{"type":"number"},
            "radius":{"type":"number"},"top_k":{"type":"integer"}
        },"required":["session","x","y","z"]}
    }},
    {"type":"function","function":{
        "name":"scene_summary",
        "description":"Return verified dataset-level statistics, including persistent-grain counts and grain-size distribution.",
        "parameters":{"type":"object","properties":{
            "session":{"type":"string","enum":["baseline","latest"]}
        },"required":["session"]}
    }},
    {"type":"function","function":{
        "name":"detect_regions",
        "description":"Discover geometry/appearance region hypotheses. These are uncalibrated hypotheses, not semantic labels.",
        "parameters":{"type":"object","properties":{
            "session":{"type":"string","enum":["baseline","latest"]},
            "dark_threshold":{"type":"number"}
        },"required":["session"]}
    }},
    {"type":"function","function":{
        "name":"region_lookup",
        "description":"Inspect a region using an exact region_name previously returned by detect_regions.",
        "parameters":{"type":"object","properties":{
            "session":{"type":"string","enum":["baseline","latest"]},
            "region_name":{"type":"string"}
        },"required":["session","region_name"]}
    }},
    {"type":"function","function":{
        "name":"load_latest_ply",
        "description":"Load a second registered PLY using the same persistent grain_id namespace.",
        "parameters":{"type":"object","properties":{"path":{"type":"string"}},"required":["path"]}
    }},
    {"type":"function","function":{
        "name":"simulate_damage",
        "description":"Create controlled in-memory synthetic damage for validation. Not real damage evidence.",
        "parameters":{"type":"object","properties":{
            "region_name":{"type":"string"},"depth":{"type":"number"},
            "radius_scale":{"type":"number"},"height_axis":{"type":"integer"}
        },"required":[]}
    }},
    {"type":"function","function":{
        "name":"compare_sessions",
        "description":"Compare baseline and latest by persistent grain_id correspondence; report missing/new IDs and centroid shifts.",
        "parameters":{"type":"object","properties":{
            "session_a":{"type":"string","enum":["baseline","latest"]},
            "session_b":{"type":"string","enum":["baseline","latest"]}
        },"required":[]}
    }},
    {"type":"function","function":{
        "name":"delta_h_damage",
        "description":"Measure registered surface-height change (latest minus baseline) on an XY grid.",
        "parameters":{"type":"object","properties":{
            "session_a":{"type":"string","enum":["baseline","latest"]},
            "session_b":{"type":"string","enum":["baseline","latest"]},
            "xy_voxel":{"type":"number"},"height_axis":{"type":"integer"},
            "height_percentile":{"type":"number"},"top_k":{"type":"integer"}
        },"required":[]}
    }},
    {"type":"function","function":{
        "name":"detect_damage",
        "description":"Detect damage using missing persistent IDs as the primary signal; report partial Gaussian-count loss separately and cluster missing grain centroids into zones.",
        "parameters":{"type":"object","properties":{
            "baseline_session":{"type":"string","enum":["baseline","latest"]},
            "latest_session":{"type":"string","enum":["baseline","latest"]},
            "partial_retention_threshold":{"type":"number"},"min_partial_gaussians":{"type":"integer"},
            "min_samples":{"type":"integer"},"eps":{"type":"number"}
        },"required":[]}
    }},
]

VALID_TOOL_NAMES = {t["function"]["name"] for t in TOOLS}
TOOL_MAP = {t["function"]["name"]: t for t in TOOLS}


def execute_tool(name, args):
    if name not in VALID_TOOL_NAMES:
        return {"ok": False, "error_type": "UNKNOWN_TOOL", "message": str(name)}
    args = args or {}
    try:
        if name == "probe_point":
            return memory.probe_point(
                args["session"],
                [args["x"], args["y"], args["z"]],
                args.get("radius", 0.15),
            )
        if name == "grain_lookup":
            return memory.grain_lookup(
                args["session"],
                args["grain_id"],
                args.get("include_indices", False),
            )
        if name == "nearest_grains":
            return memory.nearest_grains(
                args["session"],
                [args["x"], args["y"], args["z"]],
                args.get("radius", 0.25),
                args.get("top_k", 5),
            )
        if name == "scene_summary":
            return memory.scene_summary(args["session"])
        if name == "detect_regions":
            return REGION_ANALYZER.detect_regions(
                args["session"],
                args.get("dark_threshold", 0.30),
            )
        if name == "region_lookup":
            return REGION_ANALYZER.region_lookup(
                args["session"],
                args["region_name"],
            )
        if name == "load_latest_ply":
            return DAMAGE_ANALYZER.load_latest(args["path"])
        if name == "simulate_damage":
            return DAMAGE_ANALYZER.simulate_damage(
                args.get("region_name", "wheel_candidate_1"),
                args.get("depth", 0.03),
                args.get("radius_scale", 0.75),
                args.get("height_axis", 2),
            )
        if name == "compare_sessions":
            return memory.compare_sessions(
                args.get("session_a", "baseline"),
                args.get("session_b", "latest"),
            )
        if name == "delta_h_damage":
            return DAMAGE_ANALYZER.delta_h(
                args.get("session_a", "baseline"),
                args.get("session_b", "latest"),
                args.get("xy_voxel", 0.02),
                args.get("height_axis", 2),
                args.get("height_percentile", 90.0),
                3,
                args.get("top_k", 25),
            )
        if name == "detect_damage":
            return DAMAGE_ANALYZER.detect_damage(
                args.get("baseline_session", "baseline"),
                args.get("latest_session", "latest"),
                args.get("partial_retention_threshold", 0.75),
                args.get("min_partial_gaussians", 2),
                args.get("min_samples", 5),
                args.get("eps", None),
            )
        return {"ok": False, "error_type": "INTERNAL_TOOL_ERROR", "message": "Unhandled tool path"}
    except Exception as exc:
        return {"ok": False, "error_type": "TOOL_EXECUTION_ERROR", "message": str(exc)}

print("Tool API ready:", sorted(VALID_TOOL_NAMES))

Tool API ready: ['compare_sessions', 'delta_h_damage', 'detect_damage', 'detect_regions', 'grain_lookup', 'load_latest_ply', 'nearest_grains', 'probe_point', 'region_lookup', 'scene_summary', 'simulate_damage']


## 9. Grounded agent loop — routed runtime + full-tool evaluation mode


In [10]:

# ============================================================
# DESIGN GOALS
# ------------------------------------------------------------
# 1. Persistent grain_id is the sole identity source.
# 2. 3DGS opacity/scale are decoded correctly.
# 3. Semantic labels are never fabricated.
# 4. Region outputs are hypotheses, not ground truth.
# 5. Damage claims distinguish ID loss from physical proof.
# 6. Tool results are compacted before being sent back to LLM.
# 7. Provider failures are typed.
# 8. Unknown-tool recovery is conservative.
# 9. NORMAL operation can use routed tool subsets.
# 10. GENUINE LLM EVALUATION must use mode="full":
#       the LLM sees all tools and independently chooses.
#
#     run_agent(..., mode="full")
#
# ============================================================

import json
import re
import time


# ============================================================
# SYSTEM PROMPT
# ============================================================

SYSTEM_PROMPT = """
You are a grounded LEGO spatial-memory agent.

PERSISTENT IDENTITY:
- The PLY field `grain_id` is the sole persistent identity source of truth.
- Never invent grain IDs, coordinates, region names, or scene facts.

3DGS PARAMETERIZATION:
- opacity is decoded from the raw PLY opacity logit using sigmoid(opacity_logit).
- scale is decoded from raw PLY scale logs using exp(scale_log).
- Do not present raw logits or raw log-scales as physical opacity or scale.

GROUNDING:
- Use tool results for all scene-specific facts.
- Never fabricate a scene-specific value that is not supported by a tool result.
- If the required evidence is unavailable, explicitly say it is unavailable.
- Semantic labels are not present in this dataset.

TOOL ROUTING GUIDANCE:
- coordinate / XYZ query -> probe_point
- exact grain_id -> grain_lookup
- nearest / closest -> nearest_grains
- dataset counts / statistics -> scene_summary
- semantic / region question -> detect_regions
- exact known region -> region_lookup
- baseline/latest comparison -> compare_sessions
- surface-height / Delta-H -> delta_h_damage
- explicit damage-zone detection -> detect_damage
- load/register a second PLY -> load_latest_ply
- synthetic validation -> simulate_damage

REGION DISCIPLINE:
- detect_regions returns geometric hypotheses, not ground-truth semantic labels.
- Never invent region names such as region_0 or candidate_0.
- Do not convert a geometric hypothesis into a verified semantic label.

TEMPORAL / DAMAGE DISCIPLINE:
- compare_sessions establishes persistent-ID correspondence/difference.
- Missing persistent IDs are an ID-level disappearance signal.
- ID loss alone does not prove physical material removal.
- delta_h_damage provides independent registered surface-height evidence.
- detect_damage uses missing persistent IDs as its primary damage signal.
- Within-grain Gaussian-count reduction is a separate, lower-confidence signal.

ANSWER DISCIPLINE:
- Be concise and evidence-based.
- Report the actual values returned by the tools.
- Do not claim evidence that the tools did not provide.
- For detect_damage, report total missing grains, partial-loss count,
  damage-zone count, and zone details when available.
""".strip()


# ============================================================
# TOOL ROUTING MAP
# ============================================================

ROUTE_TOOLS = {
    "probe": ["probe_point"],
    "grain": ["grain_lookup"],
    "nearest": ["nearest_grains"],
    "summary": ["scene_summary"],
    "region": ["detect_regions"],
    "comparison": ["compare_sessions"],
    "delta_h": ["delta_h_damage"],
    "damage": ["detect_damage"],
    "load": ["load_latest_ply"],
    "synthetic": ["simulate_damage"],
}



SCENE_PATTERNS = [
    r"\bwhere\b",
    r"\bwhat\b",
    r"\bnear\b",
    r"\bat coordinate\b",
    r"\bgrain[_ -]?id\b",
    r"\bidentity\b",
    r"\bpersistent\b",
    r"\bconfidence\b",
    r"\bcolour\b",
    r"\bcolor\b",
    r"\bgaussian\b",
    r"\bpoint cloud\b",
    r"\bply\b",
    r"\bscene\b",
    r"\bchange\b",
    r"\bdamage\b",
    r"\bwheel\b",
    r"\bwheels\b",
    r"\bregion\b",
    r"\bpart\b",
    r"\bdelta\s*h\b",
    r"\bΔh\b",
    r"\bheight\b",
    r"\blost\b",
    r"\bsemantics?\b",
]


def is_scene_question(question):
    q = str(question).lower()
    return any(
        re.search(
            pattern,
            q,
        )
        for pattern in SCENE_PATTERNS
    )


# ============================================================
# NORMAL-USE ROUTER
#
# IMPORTANT:
# This is for normal operation only.
# It is NOT used by genuine benchmark evaluation.
#
# The ordering is intentional:
# - summary/count questions are checked before "persistent grain"
# - comparison/damage/etc. are checked before generic scene words
# ============================================================

def select_tool_names(question):
    q = str(question).lower().strip()

    # --------------------------------------------------------
    # Temporal comparison
    # --------------------------------------------------------
    if any(
        phrase in q
        for phrase in [
            "has the scene changed",
            "scene changed",
            "changed between baseline and latest",
            "compare baseline",
            "compare sessions",
            "what changed",
            "baseline versus latest",
            "baseline vs latest",
        ]
    ):
        return ROUTE_TOOLS["comparison"]

    # --------------------------------------------------------
    # Damage
    # --------------------------------------------------------
    if any(
        phrase in q
        for phrase in [
            "detect damage zones",
            "damage zones",
            "detect damage",
        ]
    ):
        return ROUTE_TOOLS["damage"]

    # --------------------------------------------------------
    # Delta-H
    # --------------------------------------------------------
    if any(
        phrase in q
        for phrase in [
            "delta h",
            "δh",
            "Δh",
            "height change",
            "surface height",
            "height difference",
        ]
    ):
        return ROUTE_TOOLS["delta_h"]

    # --------------------------------------------------------
    # Load/register latest
    # --------------------------------------------------------
    if any(
        phrase in q
        for phrase in [
            "load latest",
            "load second",
            "register second",
            "second ply",
            "latest ply",
        ]
    ):
        return ROUTE_TOOLS["load"]

    # --------------------------------------------------------
    # DATASET SUMMARY
    #
    # This MUST come before generic "persistent grain" matching.
    # --------------------------------------------------------
    if any(
        phrase in q
        for phrase in [
            "how many",
            "count",
            "counts",
            "dataset statistics",
            "dataset stats",
            "scene summary",
            "summary",
            "number of persistent grains",
            "number of grains",
            "total grains",
            "persistent grains in the",
            "persistent grains are in",
            "how many persistent grains",
        ]
    ):
        return ROUTE_TOOLS["summary"]

    # --------------------------------------------------------
    # Region / semantic
    # --------------------------------------------------------
    if any(
        word in q
        for word in [
            "wheel",
            "wheels",
            "region",
            "semantic",
            "part",
        ]
    ):
        return ROUTE_TOOLS["region"]

    # --------------------------------------------------------
    # Nearest / closest
    # --------------------------------------------------------
    if any(
        word in q
        for word in [
            "nearest",
            "closest",
        ]
    ):
        return ROUTE_TOOLS["nearest"]

    # --------------------------------------------------------
    # Exact grain ID / persistent grain
    # --------------------------------------------------------
    if (
        "grain_id" in q
        or "grain id" in q
        or "persistent grain" in q
    ):
        if any(
            word in q
            for word in [
                "coordinate",
                "at ",
                "near ",
                "where ",
            ]
        ):
            return ROUTE_TOOLS["probe"]

        return ROUTE_TOOLS["grain"]

    # --------------------------------------------------------
    # Coordinate / XYZ
    # --------------------------------------------------------
    if any(
        word in q
        for word in [
            "coordinate",
            "xyz",
            "where",
        ]
    ):
        return ROUTE_TOOLS["probe"]

    # --------------------------------------------------------
    # Safe normal-operation fallback
    # --------------------------------------------------------
    return ROUTE_TOOLS["summary"]


# ============================================================
# RESULT COMPACTION
# ============================================================

def compact_result(
    name,
    result,
    max_chars=9000,
):
    """
    Compact large tool outputs before sending them back to the
    LLM while preserving the evidence needed to answer the query.
    The original underlying result remains in the trace.
    """

    if not isinstance(
        result,
        dict,
    ):
        return result

    r = dict(
        result
    )

    # --------------------------------------------------------
    # DETECT DAMAGE
    # --------------------------------------------------------
    if name == "detect_damage":

        compact = {
            "ok": result.get(
                "ok"
            ),
            "damage_detected": result.get(
                "damage_detected"
            ),
            "total_missing_grains": result.get(
                "total_missing_grains",
                result.get(
                    "missing_grains",
                    0,
                ),
            ),
            "partially_reduced_grains": result.get(
                "partially_reduced_grains",
                0,
            ),
            "affected_grain_count": result.get(
                "affected_grain_count",
                0,
            ),
            "damage_zones_count": result.get(
                "damage_zones_count",
                0,
            ),
            "damage_zones": [],
            "partial_grain_loss_sample": [],
            "granularity_warning": result.get(
                "granularity_warning"
            ),
            "partial_retention_threshold": result.get(
                "partial_retention_threshold"
            ),
            "min_partial_gaussians": result.get(
                "min_partial_gaussians"
            ),
        }

        for zone in (
            result.get(
                "damage_zones",
                [],
            )
            or []
        ):
            compact[
                "damage_zones"
            ].append(
                {
                    "cluster_id": zone.get(
                        "cluster_id"
                    ),
                    "grain_count": zone.get(
                        "grain_count"
                    ),
                    "severity": zone.get(
                        "severity"
                    ),
                    "center": zone.get(
                        "center"
                    ),
                }
            )

        partial = result.get(
            "partial_grain_loss",
            [],
        )

        if isinstance(
            partial,
            dict,
        ):
            partial = partial.get(
                "sample",
                [],
            )

        if isinstance(
            partial,
            list,
        ):
            compact[
                "partial_grain_loss_sample"
            ] = partial[
                :5
            ]

        missing_ids = result.get(
            "missing_grain_ids",
            [],
        )

        if isinstance(
            missing_ids,
            dict,
        ):
            missing_sample = missing_ids.get(
                "sample",
                [],
            )

        elif isinstance(
            missing_ids,
            list,
        ):
            missing_sample = missing_ids[
                :10
            ]

        else:
            missing_sample = []

        compact[
            "missing_grain_id_sample"
        ] = missing_sample

        return compact

    # --------------------------------------------------------
    # COMPARE SESSIONS
    # --------------------------------------------------------
    if name == "compare_sessions":

        return {
            "ok": result.get(
                "ok"
            ),
            "session_a": result.get(
                "session_a"
            ),
            "session_b": result.get(
                "session_b"
            ),
            "persistent_grains_a": result.get(
                "persistent_grains_a"
            ),
            "persistent_grains_b": result.get(
                "persistent_grains_b"
            ),
            "common_grains": result.get(
                "common_grains"
            ),
            "lost_grain_ids": result.get(
                "lost_grain_ids",
                0,
            ),
            "new_grain_ids": result.get(
                "new_grain_ids",
                0,
            ),
            "baseline_id_retention": result.get(
                "baseline_id_retention"
            ),
            "latest_id_overlap": result.get(
                "latest_id_overlap"
            ),
            "median_centroid_shift": result.get(
                "median_centroid_shift"
            ),
            "p95_centroid_shift": result.get(
                "p95_centroid_shift"
            ),
            "median_scale_change": result.get(
                "median_scale_change"
            ),
            "median_opacity_change": result.get(
                "median_opacity_change"
            ),
            "verdict": result.get(
                "verdict"
            ),
            "registration_note": result.get(
                "registration_note"
            ),
        }

    # --------------------------------------------------------
    # GRAIN LOOKUP
    # --------------------------------------------------------
    if name == "grain_lookup":

        compact = dict(
            r
        )

        indices = compact.get(
            "vertex_indices"
        )

        if isinstance(
            indices,
            list,
        ):
            compact[
                "vertex_indices"
            ] = {
                "count": len(
                    indices
                ),
                "sample": indices[
                    :10
                ],
            }

        return compact

    # --------------------------------------------------------
    # PROBE POINT
    # Keep only compact spatial evidence.
    # --------------------------------------------------------
    if name == "probe_point":

        return {
            "ok": result.get(
                "ok"
            ),
            "occupied": result.get(
                "occupied"
            ),
            "query_xyz": result.get(
                "query_xyz"
            ),
            "neighbour_count": result.get(
                "neighbour_count"
            ),
            "unique_grain_ids": result.get(
                "unique_grain_ids"
            ),
            "grain_consistency_confidence": result.get(
                "grain_consistency_confidence"
            ),
            "normalized_entropy": result.get(
                "normalized_entropy"
            ),
            "dominant_grain_id": result.get(
                "dominant_grain_id"
            ),
            "dominant_grain_fraction": result.get(
                "dominant_grain_fraction"
            ),
            "semantic_status": result.get(
                "semantic_status"
            ),
            "mean_colour": result.get(
                "mean_colour"
            ),
            "centroid_of_neighbours": result.get(
                "centroid_of_neighbours"
            ),
            "mean_opacity": result.get(
                "mean_opacity"
            ),
            "opacity_parameterization": result.get(
                "opacity_parameterization"
            ),
            "mean_scale": result.get(
                "mean_scale"
            ),
            "scale_parameterization": result.get(
                "scale_parameterization"
            ),
        }

    # --------------------------------------------------------
    # SCENE SUMMARY
    # --------------------------------------------------------
    if name == "scene_summary":

        keys = [
            "ok",
            "session",
            "source_file",
            "source_kind",
            "gaussian_count",
            "unique_persistent_grain_count",
            "grain_id_min",
            "grain_id_max",
            "mean_gaussians_per_grain",
            "median_gaussians_per_grain",
            "max_gaussians_per_grain",
            "one_gaussian_grain_fraction",
            "heavy_tail_max_to_median",
            "semantic_regions_available",
            "semantic_note",
        ]

        return {
            key: r[key]
            for key in keys
            if key in r
        }

    # --------------------------------------------------------
    # REGION DETECTION
    # --------------------------------------------------------
    if name == "detect_regions":

        keys = [
            "ok",
            "session",
            "region_count",
            "wheel_candidate_count",
            "regions",
            "wheel_candidates",
            "semantic_status",
            "semantic_note",
        ]

        compact = {
            key: r[key]
            for key in keys
            if key in r
        }

        return compact

    # --------------------------------------------------------
    # ΔH
    # --------------------------------------------------------
    if name == "delta_h_damage":

        keys = [
            "ok",
            "method",
            "height_axis",
            "height_percentile",
            "xy_voxel",
            "median_delta_h",
            "mad_delta_h",
            "damage_threshold",
            "common_cells",
            "flagged_damage_cells",
            "median_absolute_delta_h",
            "max_negative_delta_h",
            "interpretation",
        ]

        return {
            key: r[key]
            for key in keys
            if key in r
        }

    # --------------------------------------------------------
    # GENERIC FALLBACK
    # --------------------------------------------------------
    try:

        encoded = json.dumps(
            r,
            ensure_ascii=False,
            default=str,
        )

        if len(
            encoded
        ) <= max_chars:
            return r

    except Exception:
        return r

    evidence_keys = [
        "ok",
        "session",
        "grain_id",
        "gaussian_count",
        "unique_persistent_grain_count",
        "region_count",
        "wheel_candidate_count",
        "damage_detected",
        "total_missing_grains",
        "partially_reduced_grains",
        "damage_zones_count",
    ]

    return {
        key: r[key]
        for key in evidence_keys
        if key in r
    }


# ============================================================
# PROVIDER ERROR CLASSIFIER
# ============================================================

def classify_provider_error(exc):

    text = str(
        exc
    ).lower()

    if any(
        token in text
        for token in [
            "rate limit",
            "rate_limit_exceeded",
            "429",
            "tokens per minute",
            "tpm",
        ]
    ):
        return "PROVIDER_RATE_LIMIT"

    if (
        "request too large" in text
        or "413" in text
    ):
        return "REQUEST_TOO_LARGE"

    if "timeout" in text:
        return "PROVIDER_TIMEOUT"

    return "PROVIDER_ERROR"


# ============================================================
# CONSERVATIVE UNKNOWN-TOOL RECOVERY
# ============================================================

def extract_requested_tool_from_error(
    error_text
):
    """
    Only recover a tool when the provider explicitly names the
    attempted tool. Do NOT scan arbitrary error text for tool
    names, because that could incorrectly expand the tool set.
    """

    patterns = [
        r"attempted to call tool ['\"]([^'\"]+)['\"]",
        r"tool ['\"]([^'\"]+)['\"] was not",
        r"unknown tool ['\"]([^'\"]+)['\"]",
        r"invalid tool ['\"]([^'\"]+)['\"]",
    ]

    text = str(
        error_text
    )

    for pattern in patterns:

        match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE,
        )

        if match:
            return match.group(
                1
            )

    return None


# ============================================================
# RUN AGENT
# ============================================================

def run_agent(
    question,
    max_turns=4,
    verbose=True,
    mode="routed",
):
    """
    mode="routed":
        Normal operation. A deterministic router provides a
        compact tool subset.

    mode="full":
        GENUINE LLM EVALUATION. All valid tools are exposed.
        The LLM independently selects tools using tool_choice="auto".
        No expected benchmark tool is injected.

    The complete raw tool trace is returned for evaluation.
    """

    # --------------------------------------------------------
    # TOOL AVAILABILITY
    # --------------------------------------------------------
    if mode == "full":

        active_names = list(
            VALID_TOOL_NAMES
        )

    elif mode == "routed":

        active_names = list(
            select_tool_names(
                question
            )
        )

    else:

        raise ValueError(
            "mode must be 'routed' or 'full'"
        )

    active_names = list(
        dict.fromkeys(
            active_names
        )
    )

    # Keep only registered tools.
    active_names = [
        name
        for name in active_names
        if name in VALID_TOOL_NAMES
        and name in TOOL_MAP
    ]

    if not active_names:
        raise RuntimeError(
            "No valid tools available for this request."
        )

    # --------------------------------------------------------
    # INITIAL MESSAGES
    # --------------------------------------------------------
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": question,
        },
    ]

    trace = []

    # --------------------------------------------------------
    # TURN LOOP
    # --------------------------------------------------------
    for turn in range(
        max_turns
    ):

        active_tools = [
            TOOL_MAP[name]
            for name in active_names
        ]

        # Preserve system + user + recent conversation.
        if len(
            messages
        ) > 8:
            api_messages = (
                messages[
                    :2
                ]
                + messages[
                    -6:
                ]
            )
        else:
            api_messages = messages

        last_exc = None

        # ----------------------------------------------------
        # PROVIDER RETRY LOOP
        # ----------------------------------------------------
        for attempt in range(
            EVAL_RETRIES_429 + 1
        ):

            try:

                resp = (
                    client.chat.completions.create(
                        model=MODEL,
                        max_tokens=EVAL_MAX_TOKENS,
                        temperature=0,
                        tool_choice="auto",
                        parallel_tool_calls=False,
                        messages=api_messages,
                        tools=active_tools,
                    )
                )

                last_exc = None

                break

            except Exception as exc:

                last_exc = exc

                kind = classify_provider_error(
                    exc
                )

                if (
                    kind
                    == "PROVIDER_RATE_LIMIT"
                    and attempt
                    < EVAL_RETRIES_429
                ):

                    time.sleep(
                        EVAL_BACKOFF_SECONDS
                        * (
                            attempt
                            + 1
                        )
                    )

                    continue

                break

        # ----------------------------------------------------
        # PROVIDER ERROR
        # ----------------------------------------------------
        if last_exc is not None:

            error_text = str(
                last_exc
            )

            failure_type = (
                classify_provider_error(
                    last_exc
                )
            )

            # ----------------------------------------------
            # Conservative unknown-tool recovery.
            #
            # Only add a tool if the provider explicitly
            # reports the attempted tool.
            # ----------------------------------------------
            requested_tool = (
                extract_requested_tool_from_error(
                    error_text
                )
            )

            if (
                requested_tool
                and requested_tool
                in VALID_TOOL_NAMES
                and requested_tool
                not in active_names
            ):

                active_names.append(
                    requested_tool
                )

                if verbose:

                    print(
                        "Provider requested previously "
                        f"unavailable tool: {requested_tool}"
                    )

                continue

            return {
                "answer": "",
                "trace": trace,
                "provider_failure": failure_type,
                "provider_error": error_text,
                "safe_refusal": False,
            }

        # ----------------------------------------------------
        # MODEL MESSAGE
        # ----------------------------------------------------
        msg = (
            resp
            .choices[
                0
            ]
            .message
        )

        messages.append(
            msg.model_dump(
                exclude_none=True
            )
        )

        # ----------------------------------------------------
        # FINAL TEXT RESPONSE
        # ----------------------------------------------------
        if not msg.tool_calls:

            final = (
                msg.content
                or ""
            )

            if (
                is_scene_question(
                    question
                )
                and not trace
                and not final.strip()
            ):

                return {
                    "answer": (
                        "[UNGROUNDED RESPONSE BLOCKED]"
                    ),
                    "trace": trace,
                    "provider_failure": None,
                    "safe_refusal": True,
                }

            if (
                is_scene_question(
                    question
                )
                and not trace
                and final.strip()
                == "[UNGROUNDED RESPONSE BLOCKED]"
            ):

                return {
                    "answer": final,
                    "trace": trace,
                    "provider_failure": None,
                    "safe_refusal": True,
                }

            return {
                "answer": final,
                "trace": trace,
                "provider_failure": None,
                "safe_refusal": False,
            }

        # ----------------------------------------------------
        # TOOL CALLS
        # ----------------------------------------------------
        for tc in msg.tool_calls:

            name = (
                tc.function.name
            )

            raw_arguments = (
                tc.function.arguments
            )

            try:

                args = json.loads(
                    raw_arguments
                )

                if not isinstance(
                    args,
                    dict,
                ):

                    raise ValueError(
                        "Tool arguments must decode to an object."
                    )

                if (
                    name
                    not in VALID_TOOL_NAMES
                ):

                    result = {
                        "ok": False,
                        "error_type": "UNKNOWN_TOOL",
                        "message": (
                            "Model requested an unregistered tool: "
                            + str(name)
                        ),
                    }

                else:

                    # In FULL mode, do not silently expand or
                    # substitute tools here. The tool was already
                    # available to the LLM.
                    #
                    # In ROUTED mode, this permits a model-requested
                    # tool if the provider/model selected it.
                    if (
                        name
                        not in active_names
                    ):
                        active_names.append(
                            name
                        )

                    result = execute_tool(
                        name,
                        args,
                    )

            except json.JSONDecodeError as exc:

                args = {}

                result = {
                    "ok": False,
                    "error_type": "INVALID_TOOL_ARGUMENTS_JSON",
                    "message": str(
                        exc
                    ),
                }

            except Exception as exc:

                args = {}

                result = {
                    "ok": False,
                    "error_type": "TOOL_EXECUTION_ERROR",
                    "message": str(
                        exc
                    ),
                }

            # ------------------------------------------------
            # COMPLETE TRACE
            # ------------------------------------------------
            trace.append(
                {
                    "tool": name,
                    "args": args,
                    "result": result,
                    "turn": turn,
                }
            )

            # ------------------------------------------------
            # OPTIONAL DEBUG PRINT
            #
            # Cell 11 calls run_agent(verbose=False), so these
            # do not appear in the recruiter notebook.
            # ------------------------------------------------
            if verbose:

                print(
                    f"TOOL {name} -> "
                    f"{compact_result(name, result)}"
                )

            # ------------------------------------------------
            # SEND COMPACT RESULT BACK TO MODEL
            # ------------------------------------------------
            compacted = compact_result(
                name,
                result,
            )

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": json.dumps(
                        compacted,
                        default=str,
                        ensure_ascii=False,
                    ),
                }
            )

    # --------------------------------------------------------
    # MAX TURNS
    # --------------------------------------------------------
    return {
        "answer": "[MAX_TURNS_REACHED]",
        "trace": trace,
        "provider_failure": "MAX_TURNS_REACHED",
        "safe_refusal": False,
    }


# ============================================================
# READY
# ============================================================

print(
    "Grounded agent ready:",
    MODEL
)

print(
    "Evaluation mode:",
    "mode='full' exposes all tools; the LLM selects tools autonomously."
)

Grounded agent ready: openai/gpt-oss-120b
Evaluation mode: mode='full' exposes all tools; the LLM selects tools autonomously.


## 10. Deterministic tool validation (not an agent accuracy score)

In [11]:
DETERMINISTIC_RESULTS = {}

DETERMINISTIC_RESULTS["probe_point"] = execute_tool(
    "probe_point",
    {
        "session": "baseline",
        "x": -0.29385,
        "y": -0.76548,
        "z": -0.31067,
        "radius": 0.02,
    },
)
DETERMINISTIC_RESULTS["grain_lookup"] = execute_tool(
    "grain_lookup",
    {
        "session": "baseline",
        "grain_id": 16147,
    },
)
DETERMINISTIC_RESULTS["scene_summary"] = execute_tool(
    "scene_summary",
    {"session": "baseline"},
)
DETERMINISTIC_RESULTS["compare_sessions"] = execute_tool(
    "compare_sessions",
    {"session_a": "baseline", "session_b": "latest"},
)
DETERMINISTIC_RESULTS["detect_damage"] = execute_tool(
    "detect_damage",
    {"baseline_session": "baseline", "latest_session": "latest"},
)
DETERMINISTIC_RESULTS["delta_h_damage"] = execute_tool(
    "delta_h_damage",
    {"session_a": "baseline", "session_b": "latest", "xy_voxel": 0.02, "height_axis": 2},
)

print("Deterministic tool checks:")
for name, result in DETERMINISTIC_RESULTS.items():
    print(
        f"{name:20s}",
        "PASS" if isinstance(result, dict) and result.get("ok") else "FAIL",
    )

assert DETERMINISTIC_RESULTS["scene_summary"]["unique_persistent_grain_count"] == 35731
assert DETERMINISTIC_RESULTS["compare_sessions"]["lost_grain_ids"] == 2117
assert DETERMINISTIC_RESULTS["compare_sessions"]["new_grain_ids"] == 0
assert DETERMINISTIC_RESULTS["detect_damage"]["total_missing_grains"] == 2117
assert DETERMINISTIC_RESULTS["detect_damage"]["partially_reduced_grains"] == 160
assert DETERMINISTIC_RESULTS["detect_damage"]["damage_zones_count"] == 2
assert DETERMINISTIC_RESULTS["delta_h_damage"]["flagged_damage_cells"] == 122
print("Deterministic validation assertions passed.")

Deterministic tool checks:
probe_point          PASS
grain_lookup         PASS
scene_summary        PASS
compare_sessions     PASS
detect_damage        PASS
delta_h_damage       PASS
Deterministic validation assertions passed.


## 11. Genuine LLM evaluation

In [12]:
# IMPORTANT:
# - Uses the REAL run_agent() implementation.
# - Uses mode="full" for genuine tool selection.
# - The expected tool is NEVER passed to run_agent().
# - All registered tools are available to the LLM.
# - tool_choice="auto" determines what the LLM actually calls.
# - The evaluator checks the resulting trace afterward.
# - Provider failures are not converted into FALSE tool calls.
# - Raw TOOL(...) diagnostics are hidden from the notebook.
#
# This cell evaluates:
#   1. Tool selection
#   2. Tool argument correctness
#   3. Expected tool result validity
#   4. Grounded answer correctness
# ============================================================

import io
import time
import contextlib
import pandas as pd


# ------------------------------------------------------------
# REQUIRE REAL AGENT
# ------------------------------------------------------------
if "run_agent" not in globals() or not callable(run_agent):
    raise RuntimeError(
        "run_agent() is not defined. "
        "Run the grounded-agent cell before Cell 11."
    )


# ------------------------------------------------------------
# BENCHMARK
# ------------------------------------------------------------
AGENT_TEST_SET = [
    (
        "L01",
        "What persistent grain identity is at coordinate "
        "(-0.29385, -0.76548, -0.31067) in the baseline scan "
        "with a search radius of 0.02?",
        "probe_point",
    ),
    (
        "L02",
        "Tell me about persistent grain_id 16147.",
        "grain_lookup",
    ),
    (
        "L03",
        "How many persistent grains are in the baseline LEGO PLY?",
        "scene_summary",
    ),
    (
        "L04",
        "What semantic region is represented in the supplied LEGO PLY?",
        "detect_regions",
    ),
    (
        "L05",
        "Has the LEGO scene changed between baseline and latest?",
        "compare_sessions",
    ),
    (
        "L06",
        "Detect damage zones by comparing baseline and latest sessions.",
        "detect_damage",
    ),
]


# ------------------------------------------------------------
# NORMALIZE AGENT OUTPUT
# ------------------------------------------------------------
def normalize_run_agent_output(raw):

    if not isinstance(raw, dict):
        return {
            "answer": str(raw),
            "trace": [],
            "provider_failure": None,
            "safe_refusal": False,
        }

    trace = raw.get(
        "trace",
        [],
    )

    if not isinstance(trace, list):
        trace = []

    return {
        "answer": str(
            raw.get(
                "answer",
                "",
            )
        ),
        "trace": trace,
        "provider_failure": raw.get(
            "provider_failure"
        ),
        "safe_refusal": bool(
            raw.get(
                "safe_refusal",
                False,
            )
        ),
    }


# ------------------------------------------------------------
# NORMALIZE TRACE
# ------------------------------------------------------------
def normalize_trace(trace):

    normalized = []

    for item in trace:

        if not isinstance(
            item,
            dict,
        ):
            continue

        normalized.append(
            {
                "name": (
                    item.get("tool")
                    or item.get("name")
                    or item.get("function")
                ),
                "arguments": (
                    item.get("args")
                    or item.get("arguments")
                    or {}
                ),
                "result": item.get(
                    "result"
                ),
                "turn": item.get(
                    "turn"
                ),
            }
        )

    return normalized


# ------------------------------------------------------------
# PROVIDER FAILURE CLASSIFIER
# ------------------------------------------------------------
def classify_provider_failure(value):

    if value is None:
        return None

    text = str(
        value
    ).lower()

    if any(
        token in text
        for token in [
            "429",
            "rate limit",
            "rate_limit_exceeded",
            "tokens per minute",
            "tpm",
        ]
    ):
        return "PROVIDER_RATE_LIMIT"

    if "timeout" in text:
        return "PROVIDER_TIMEOUT"

    if (
        "request too large" in text
        or "413" in text
    ):
        return "REQUEST_TOO_LARGE"

    if any(
        token in text
        for token in [
            "provider error",
            "api error",
            "service unavailable",
        ]
    ):
        return "PROVIDER_ERROR"

    return None


# ------------------------------------------------------------
# EXPECTED TOOL ARGUMENT VALIDATION
# ------------------------------------------------------------
def validate_tool_arguments(
    case_id,
    tool_calls,
):

    if case_id == "L01":

        calls = [
            c
            for c in tool_calls
            if c["name"] == "probe_point"
        ]

        if not calls:
            return False

        args = calls[-1]["arguments"]

        try:

            return (
                args.get("session")
                == "baseline"
                and abs(
                    float(args.get("x"))
                    - (-0.29385)
                ) < 1e-8
                and abs(
                    float(args.get("y"))
                    - (-0.76548)
                ) < 1e-8
                and abs(
                    float(args.get("z"))
                    - (-0.31067)
                ) < 1e-8
                and abs(
                    float(args.get("radius"))
                    - 0.02
                ) < 1e-8
            )

        except Exception:
            return False


    elif case_id == "L02":

        calls = [
            c
            for c in tool_calls
            if c["name"] == "grain_lookup"
        ]

        if not calls:
            return False

        return any(
            call["arguments"].get(
                "grain_id"
            ) == 16147
            and call["arguments"].get(
                "session"
            ) == "baseline"
            for call in calls
        )


    elif case_id == "L03":

        calls = [
            c
            for c in tool_calls
            if c["name"] == "scene_summary"
        ]

        if not calls:
            return False

        return any(
            call["arguments"].get(
                "session"
            ) == "baseline"
            for call in calls
        )


    elif case_id == "L04":

        calls = [
            c
            for c in tool_calls
            if c["name"] == "detect_regions"
        ]

        if not calls:
            return False

        return any(
            call["arguments"].get(
                "session"
            ) == "baseline"
            for call in calls
        )


    elif case_id == "L05":

        calls = [
            c
            for c in tool_calls
            if c["name"] == "compare_sessions"
        ]

        if not calls:
            return False

        return any(
            call["arguments"].get(
                "session_a",
                "baseline",
            ) == "baseline"
            and call["arguments"].get(
                "session_b",
                "latest",
            ) == "latest"
            for call in calls
        )


    elif case_id == "L06":

        calls = [
            c
            for c in tool_calls
            if c["name"] == "detect_damage"
        ]

        if not calls:
            return False

        return any(
            call["arguments"].get(
                "baseline_session",
                "baseline",
            ) == "baseline"
            and call["arguments"].get(
                "latest_session",
                "latest",
            ) == "latest"
            for call in calls
        )

    return False


# ------------------------------------------------------------
# EXPECTED TOOL SUCCESSFUL RESULT
# ------------------------------------------------------------
def get_successful_expected_results(
    expected_tool,
    tool_calls,
):

    return [
        call["result"]
        for call in tool_calls
        if (
            call["name"] == expected_tool
            and isinstance(
                call.get("result"),
                dict,
            )
            and call["result"].get(
                "ok"
            ) is True
        )
    ]


# ------------------------------------------------------------
# GROUNDED ANSWER VALIDATION
# ------------------------------------------------------------
def evaluate_grounded_answer(
    case_id,
    answer,
    tool_calls,
):

    if not answer or not str(answer).strip():
        return False

    answer = str(
        answer
    )

    answer_norm = answer.lower()

    answer_digits = (
        answer
        .replace(",", "")
        .replace("\u202f", "")
        .replace(" ", "")
    )

    # --------------------------------------------------------
    # Only successful expected-tool results are evidence.
    # --------------------------------------------------------
    expected_tool = {
        "L01": "probe_point",
        "L02": "grain_lookup",
        "L03": "scene_summary",
        "L04": "detect_regions",
        "L05": "compare_sessions",
        "L06": "detect_damage",
    }.get(
        case_id
    )

    successful_calls = get_successful_expected_results(
        expected_tool,
        tool_calls,
    )

    if not successful_calls:
        return False


    # --------------------------------------------------------
    # L01
    # --------------------------------------------------------
    if case_id == "L01":

        result = successful_calls[-1]

        gid = result.get(
            "dominant_grain_id"
        )

        return (
            gid is not None
            and str(gid)
            in answer_digits
        )


    # --------------------------------------------------------
    # L02
    # --------------------------------------------------------
    elif case_id == "L02":

        valid_results = [
            result
            for result in successful_calls
            if result.get(
                "grain_id"
            ) == 16147
        ]

        if not valid_results:
            return False

        return (
            "16147" in answer_digits
            and "centroid" in answer_norm
            and "opacity" in answer_norm
            and "scale" in answer_norm
        )


    # --------------------------------------------------------
    # L03
    # --------------------------------------------------------
    elif case_id == "L03":

        result = successful_calls[-1]

        count = result.get(
            "unique_persistent_grain_count"
        )

        return (
            count is not None
            and str(count)
            in answer_digits
        )


    # --------------------------------------------------------
    # L04
    # --------------------------------------------------------
    elif case_id == "L04":

        # The answer must correctly preserve the absence of
        # semantic ground truth.
        return (
            "semantic" in answer_norm
            and (
                "ground-truth" in answer_norm
                or "ground truth" in answer_norm
                or "geometric" in answer_norm
                or "hypoth" in answer_norm
                or "not verified" in answer_norm
                or "not available" in answer_norm
                or "no semantic" in answer_norm
            )
        )


    # --------------------------------------------------------
    # L05
    # --------------------------------------------------------
    elif case_id == "L05":

        result = successful_calls[-1]

        lost = int(
            result.get(
                "lost_grain_ids",
                0,
            )
        )

        new = int(
            result.get(
                "new_grain_ids",
                0,
            )
        )

        conclusion_present = any(
            phrase in answer_norm
            for phrase in [
                "persistent-id difference",
                "persistent id difference",
                "persistent-id change",
                "persistent id change",
                "scene has changed",
                "scene changed",
                "scene shows a change",
                "id-level difference",
                "id-level change",
                "change between",
                "difference between",
                "has changed",
            ]
        )

        unsupported_physical_claim = any(
            phrase in answer_norm
            for phrase in [
                "physical damage is confirmed",
                "physical damage confirmed",
                "material was definitely removed",
                "material loss is confirmed",
                "physical removal is confirmed",
            ]
        )

        return (
            str(lost)
            in answer_digits
            and str(new)
            in answer_digits
            and conclusion_present
            and not unsupported_physical_claim
        )


    # --------------------------------------------------------
    # L06
    # --------------------------------------------------------
    elif case_id == "L06":

        result = successful_calls[-1]

        missing = int(
            result.get(
                "total_missing_grains",
                result.get(
                    "missing_grains",
                    0,
                ),
            )
        )

        partial = int(
            result.get(
                "partially_reduced_grains",
                0,
            )
        )

        zones = int(
            result.get(
                "damage_zones_count",
                0,
            )
        )

        return (
            "damage" in answer_norm
            and "zone" in answer_norm
            and str(missing)
            in answer_digits
            and str(partial)
            in answer_digits
            and str(zones)
            in answer_digits
        )


    return False


# ------------------------------------------------------------
# SINGLE CASE EVALUATION
# ------------------------------------------------------------
def evaluate_agent_case(
    case_id,
    question,
    expected_tool,
):

    start = time.perf_counter()

    try:

        # ----------------------------------------------------
        # GENUINE LLM CALL
        #
        # mode="full" is intentional.
        # All tools are available to the model.
        # The benchmark's expected tool is NOT supplied.
        # ----------------------------------------------------
        captured_stdout = io.StringIO()

        with contextlib.redirect_stdout(
            captured_stdout
        ):

            raw = run_agent(
                question,
                max_turns=4,
                verbose=False,
                mode="full",
            )

        normalized = (
            normalize_run_agent_output(
                raw
            )
        )

        answer = normalized[
            "answer"
        ]

        tool_calls = normalize_trace(
            normalized[
                "trace"
            ]
        )

        actual_tools = [
            call["name"]
            for call in tool_calls
            if call.get("name")
        ]

        tool_executed = (
            expected_tool
            in actual_tools
        )

        # ----------------------------------------------------
        # Provider failure reported directly by run_agent
        # ----------------------------------------------------
        provider_failure = (
            classify_provider_failure(
                normalized.get(
                    "provider_failure"
                )
            )
        )

        # ----------------------------------------------------
        # Tool result validity for EXPECTED tool only.
        # ----------------------------------------------------
        expected_results = [
            call
            for call in tool_calls
            if (
                call["name"]
                == expected_tool
                and isinstance(
                    call.get("result"),
                    dict,
                )
            )
        ]

        tool_result_check = (
            bool(
                expected_results
                and any(
                    result["result"].get(
                        "ok"
                    ) is True
                    for result in expected_results
                )
            )
        )

        # ----------------------------------------------------
        # Tool arguments for EXPECTED tool only.
        # ----------------------------------------------------
        tool_argument_check = (
            validate_tool_arguments(
                case_id,
                tool_calls,
            )
            if tool_executed
            else False
        )

        # ----------------------------------------------------
        # If no direct provider failure was returned, inspect
        # only provider_error metadata.
        #
        # Do NOT classify normal scene answers/results as provider
        # failures based on arbitrary text content.
        # ----------------------------------------------------
        if provider_failure is None:

            provider_failure = (
                classify_provider_failure(
                    normalized.get(
                        "provider_error"
                    )
                )
            )

        # ----------------------------------------------------
        # Provider unavailable:
        # tool execution facts are preserved.
        # Answer is NOT scored.
        # ----------------------------------------------------
        if provider_failure:

            return {
                "id": case_id,
                "question": question,
                "answer": answer,
                "expected_tool": expected_tool,
                "actual_tools": actual_tools,
                "tool_executed": tool_executed,
                "tool_result_check": (
                    tool_result_check
                ),
                "tool_argument_check": (
                    tool_argument_check
                ),
                "answer_check": None,
                "status": "UNAVAILABLE",
                "safe_refusal": False,
                "provider_failure": provider_failure,
                "tool_call_count": len(
                    actual_tools
                ),
                "latency_s": round(
                    time.perf_counter()
                    - start,
                    3,
                ),
            }

        # ----------------------------------------------------
        # Grounded answer
        # ----------------------------------------------------
        answer_check = (
            evaluate_grounded_answer(
                case_id,
                answer,
                tool_calls,
            )
        )

        # ----------------------------------------------------
        # Safe refusal
        # ----------------------------------------------------
        safe_refusal = bool(
            normalized.get(
                "safe_refusal",
                False,
            )
        )

        if not safe_refusal:

            safe_refusal = (
                answer.strip()
                == "[UNGROUNDED RESPONSE BLOCKED]"
            )

        # ----------------------------------------------------
        # Final status
        # ----------------------------------------------------
        if (
            safe_refusal
            and not tool_executed
        ):

            status = "SAFE_REFUSAL"

        elif (
            tool_executed
            and tool_result_check
            and tool_argument_check
            and answer_check
        ):

            status = "PASS"

        else:

            status = "FAIL"

        return {
            "id": case_id,
            "question": question,
            "answer": answer,
            "expected_tool": expected_tool,
            "actual_tools": actual_tools,
            "tool_executed": tool_executed,
            "tool_result_check": (
                tool_result_check
            ),
            "tool_argument_check": (
                tool_argument_check
            ),
            "answer_check": bool(
                answer_check
            ),
            "status": status,
            "safe_refusal": safe_refusal,
            "provider_failure": None,
            "tool_call_count": len(
                actual_tools
            ),
            "latency_s": round(
                time.perf_counter()
                - start,
                3,
            ),
        }


    except Exception as exc:

        provider_failure = (
            classify_provider_failure(
                exc
            )
        )

        if provider_failure:

            return {
                "id": case_id,
                "question": question,
                "answer": "[PROVIDER UNAVAILABLE]",
                "expected_tool": expected_tool,
                "actual_tools": [],
                "tool_executed": False,
                "tool_result_check": None,
                "tool_argument_check": None,
                "answer_check": None,
                "status": "UNAVAILABLE",
                "safe_refusal": False,
                "provider_failure": provider_failure,
                "tool_call_count": 0,
                "latency_s": round(
                    time.perf_counter()
                    - start,
                    3,
                ),
            }

        return {
            "id": case_id,
            "question": question,
            "answer": "[AGENT EVALUATION ERROR]",
            "expected_tool": expected_tool,
            "actual_tools": [],
            "tool_executed": False,
            "tool_result_check": False,
            "tool_argument_check": False,
            "answer_check": False,
            "status": "FAIL",
            "safe_refusal": False,
            "provider_failure": None,
            "tool_call_count": 0,
            "latency_s": round(
                time.perf_counter()
                - start,
                3,
            ),
        }


# ------------------------------------------------------------
# RUN BENCHMARK
# ------------------------------------------------------------
AGENT_RESULTS = [
    evaluate_agent_case(
        case_id,
        question,
        expected_tool,
    )
    for (
        case_id,
        question,
        expected_tool,
    ) in AGENT_TEST_SET
]

agent_df = pd.DataFrame(
    AGENT_RESULTS
)


# ------------------------------------------------------------
# CLEAN DISPLAY ANSWER
# ------------------------------------------------------------
def clean_display_answer(
    text,
    max_chars=700,
):

    text = (
        str(text)
        .replace("\n", " ")
        .replace("\r", " ")
        .strip()
    )

    if len(text) <= max_chars:
        return text

    return (
        text[:max_chars].rstrip()
        + " ..."
    )


display_df = agent_df.copy()

display_df["answer"] = (
    display_df[
        "answer"
    ].apply(
        clean_display_answer
    )
)


# ------------------------------------------------------------
# MAIN RECRUITER TABLE
# ------------------------------------------------------------
recruiter_columns = [
    "id",
    "question",
    "answer",
    "expected_tool",
    "actual_tools",
    "tool_executed",
    "tool_result_check",
    "tool_argument_check",
    "answer_check",
    "status",
]

print(
    "=" * 120
)

print(
    "LLM AGENT CASE RESULTS"
)

print(
    "=" * 120
)

print(
    display_df[
        recruiter_columns
    ].to_string(
        index=False,
        max_colwidth=80,
    )
)


# ------------------------------------------------------------
# CASE-BY-CASE CHECKS
# ------------------------------------------------------------
print()
print(
    "=" * 90
)

print(
    "CASE CHECKS"
)

print(
    "=" * 90
)


for _, row in display_df.iterrows():

    print()
    print(
        f"{row['id']}  |  {row['status']}"
    )

    print(
        f"Question: {row['question']}"
    )

    print(
        f"Answer:   {row['answer']}"
    )

    print(
        f"Expected tool: {row['expected_tool']}"
    )

    print(
        f"Actual tool(s): {row['actual_tools']}"
    )

    print(
        "Tool executed:     "
        + (
            "TRUE"
            if row["tool_executed"]
            else "FALSE"
        )
    )

    print(
        "Expected result:   "
        + (
            "TRUE"
            if row["tool_result_check"]
            else "FALSE"
        )
    )

    print(
        "Tool arguments:     "
        + (
            "TRUE"
            if row["tool_argument_check"]
            else "FALSE"
        )
    )

    if pd.isna(
        row["answer_check"]
    ):

        print(
            "Grounded answer:    NOT SCORED"
        )

    else:

        print(
            "Grounded answer:    "
            + (
                "TRUE"
                if row["answer_check"]
                else "FALSE"
            )
        )

    if row[
        "provider_failure"
    ]:

        print(
            "Provider:            "
            + str(
                row[
                    "provider_failure"
                ]
            )
        )


# ------------------------------------------------------------
# HONEST METRICS
# ------------------------------------------------------------
print()
print(
    "=" * 90
)

print(
    "GENUINE LLM AGENT EVALUATION SUMMARY"
)

print(
    "=" * 90
)


scored_df = display_df[
    display_df["status"]
    != "UNAVAILABLE"
].copy()

unavailable_df = display_df[
    display_df["status"]
    == "UNAVAILABLE"
].copy()

safe_refusal_df = display_df[
    display_df["status"]
    == "SAFE_REFUSAL"
].copy()


n_total = len(
    display_df
)

n_scored = len(
    scored_df
)

n_unavailable = len(
    unavailable_df
)

n_safe_refusal = len(
    safe_refusal_df
)


if n_scored > 0:

    routing_passes = int(
        scored_df[
            "tool_executed"
        ]
        .astype(bool)
        .sum()
    )

    result_passes = int(
        scored_df[
            "tool_result_check"
        ]
        .astype(bool)
        .sum()
    )

    argument_passes = int(
        scored_df[
            "tool_argument_check"
        ]
        .astype(bool)
        .sum()
    )

    answer_passes = int(
        scored_df[
            "answer_check"
        ]
        .astype(bool)
        .sum()
    )

    overall_passes = int(
        scored_df[
            [
                "tool_executed",
                "tool_result_check",
                "tool_argument_check",
                "answer_check",
            ]
        ]
        .astype(bool)
        .all(axis=1)
        .sum()
    )

else:

    routing_passes = 0
    result_passes = 0
    argument_passes = 0
    answer_passes = 0
    overall_passes = 0


def pct(
    value,
    total,
):

    return (
        100.0
        * value
        / max(
            1,
            total,
        )
    )


print(
    f"Total cases:                 "
    f"{n_total}"
)

print(
    f"Scored cases:                "
    f"{n_scored}"
)

print(
    f"Unavailable cases:           "
    f"{n_unavailable}"
)

print(
    f"Safe-refusal cases:          "
    f"{n_safe_refusal}"
)

print(
    f"Tool selection accuracy:     "
    f"{routing_passes}/{n_scored} "
    f"({pct(routing_passes, n_scored):.1f}%)"
)

print(
    f"Expected tool result:        "
    f"{result_passes}/{n_scored} "
    f"({pct(result_passes, n_scored):.1f}%)"
)

print(
    f"Tool argument accuracy:      "
    f"{argument_passes}/{n_scored} "
    f"({pct(argument_passes, n_scored):.1f}%)"
)

print(
    f"Grounded answer accuracy:    "
    f"{answer_passes}/{n_scored} "
    f"({pct(answer_passes, n_scored):.1f}%)"
)

print(
    f"Overall scored accuracy:     "
    f"{overall_passes}/{n_scored} "
    f"({pct(overall_passes, n_scored):.1f}%)"
)

print(
    f"Provider unavailable:        "
    f"{n_unavailable}/{n_total} "
    f"({pct(n_unavailable, n_total):.1f}%)"
)

print(
    f"Safe-refusal rate:           "
    f"{n_safe_refusal}/{n_total} "
    f"({pct(n_safe_refusal, n_total):.1f}%)"
)


# ------------------------------------------------------------
# PROVIDER-UNAVAILABLE CASES
# ------------------------------------------------------------
if n_unavailable > 0:

    print()
    print(
        "Unavailable cases are excluded from scored accuracy "
        "because the provider did not complete the LLM turn."
    )

    for _, row in unavailable_df.iterrows():

        print(
            f"{row['id']}: "
            f"tool_executed="
            f"{row['tool_executed']}, "
            f"tool="
            f"{row['actual_tools']}, "
            f"reason="
            f"{row['provider_failure']}"
        )

LLM AGENT CASE RESULTS
 id                                                                         question                                                                           answer    expected_tool                 actual_tools  tool_executed  tool_result_check  tool_argument_check  answer_check status
L01 What persistent grain identity is at coordinate (-0.29385, -0.76548, -0.31067... The baseline scan reports that the dominant persistent grain at the queried p...      probe_point                [probe_point]           True               True                 True          True   PASS
L02                                         Tell me about persistent grain_id 16147. **Persistent grain_id 16147**  | Property | Baseline | Latest | |----------|-...     grain_lookup [grain_lookup, grain_lookup]           True               True                 True          True   PASS
L03                         How many persistent grains are in the baseline LEGO PLY?                     The base

## 12. Evaluation metrics that do not hide provider failures

In [13]:
# ============================================================
# This cell is deliberately SEPARATE from the genuine LLM
# evaluation above.
#
# It tests the underlying tools/pipeline without an LLM.
# A 100% here means the deterministic pipeline passed its
# explicit checks; it does NOT mean the agent is 100% accurate.
# ============================================================

print(
    "=" * 68
)

print(
    "DETERMINISTIC PIPELINE VALIDATION"
)

print(
    "=" * 68
)


# ------------------------------------------------------------
# 1. BASELINE SUMMARY
# ------------------------------------------------------------
BASELINE_SUMMARY = memory.scene_summary(
    "baseline"
)

assert (
    BASELINE_SUMMARY["ok"]
    is True
), "Baseline scene summary failed."

assert (
    BASELINE_SUMMARY[
        "unique_persistent_grain_count"
    ]
    == 35731
), (
    "Unexpected baseline persistent-grain count: "
    f"{BASELINE_SUMMARY.get('unique_persistent_grain_count')}"
)


# ------------------------------------------------------------
# 2. SYNTHETIC LATEST SUMMARY
# ------------------------------------------------------------
LATEST_SUMMARY = memory.scene_summary(
    "latest"
)

assert (
    LATEST_SUMMARY["ok"]
    is True
), "Latest scene summary failed."


# ------------------------------------------------------------
# 3. PERSISTENT-ID COMPARISON
# ------------------------------------------------------------
COMPARE_RESULT = memory.compare_sessions(
    "baseline",
    "latest",
)

assert (
    COMPARE_RESULT.get("ok")
    is True
), "compare_sessions failed."

assert (
    COMPARE_RESULT.get(
        "lost_grain_ids"
    )
    == 2117
), (
    "Unexpected lost grain count: "
    f"{COMPARE_RESULT.get('lost_grain_ids')}"
)

assert (
    COMPARE_RESULT.get(
        "new_grain_ids"
    )
    == 0
), (
    "Unexpected new grain count: "
    f"{COMPARE_RESULT.get('new_grain_ids')}"
)

assert (
    abs(
        float(
            COMPARE_RESULT.get(
                "latest_id_overlap",
                1.0,
            )
        )
        - 1.0
    )
    < 1e-9
), "Latest persistent-ID overlap is not 1.0."

assert (
    abs(
        float(
            COMPARE_RESULT.get(
                "median_centroid_shift",
                0.0,
            )
        )
    )
    < 1e-9
), "Unexpected median centroid shift."


# ------------------------------------------------------------
# 4. DAMAGE DETECTION
# ------------------------------------------------------------
DETECT_RESULT = DAMAGE_ANALYZER.detect_damage(
    "baseline",
    "latest",
)

assert (
    DETECT_RESULT.get("ok")
    is True
), "detect_damage failed."

assert (
    DETECT_RESULT.get(
        "damage_detected"
    )
    is True
), "Synthetic damage was not detected."

assert (
    DETECT_RESULT.get(
        "total_missing_grains"
    )
    == 2117
), (
    "Unexpected missing-grain count: "
    f"{DETECT_RESULT.get('total_missing_grains')}"
)

assert (
    DETECT_RESULT.get(
        "partially_reduced_grains"
    )
    == 160
), (
    "Unexpected partial-loss count: "
    f"{DETECT_RESULT.get('partially_reduced_grains')}"
)

assert (
    DETECT_RESULT.get(
        "damage_zones_count"
    )
    == 2
), (
    "Unexpected damage-zone count: "
    f"{DETECT_RESULT.get('damage_zones_count')}"
)

# The primary damage-zone counts must reconcile with
# completely missing persistent IDs.
zone_missing_sum = sum(
    int(
        zone.get(
            "grain_count",
            0,
        )
    )
    for zone in DETECT_RESULT.get(
        "damage_zones",
        []
    )
)

assert (
    zone_missing_sum
    == DETECT_RESULT.get(
        "total_missing_grains"
    )
), (
    "Damage-zone grain counts do not reconcile "
    "with total missing persistent grains."
)

assert (
    DETECT_RESULT.get(
        "affected_grain_count"
    )
    == 2277
), (
    "Unexpected affected-grain count: "
    f"{DETECT_RESULT.get('affected_grain_count')}"
)


# ------------------------------------------------------------
# 5. DELTA-H INDEPENDENT CHECK
# ------------------------------------------------------------
DELTA_H_RESULT = DAMAGE_ANALYZER.delta_h(
    "baseline",
    "latest",
    xy_voxel=0.02,
    height_axis=2,
)

assert (
    DELTA_H_RESULT.get("ok")
    is True
), "delta_h_damage failed."

assert (
    DELTA_H_RESULT.get(
        "flagged_damage_cells",
        0,
    )
    > 0
), "ΔH did not flag any damage cells."


# ------------------------------------------------------------
# 6. SYNTHETIC ARTIFACT CHECK
# ------------------------------------------------------------
if "synthetic_result" in globals():

    assert (
        synthetic_result.get(
            "ok"
        )
        is True
    ), "Synthetic PLY creation failed."

    assert (
        synthetic_result.get(
            "original_gaussian_count"
        )
        == 194018
    ), (
        "Unexpected original Gaussian count: "
        f"{synthetic_result.get('original_gaussian_count')}"
    )

    assert (
        synthetic_result.get(
            "damaged_gaussian_count"
        )
        == 182928
    ), (
        "Unexpected damaged Gaussian count: "
        f"{synthetic_result.get('damaged_gaussian_count')}"
    )

    assert (
        synthetic_result.get(
            "removed_gaussians"
        )
        == 11090
    ), (
        "Unexpected removed Gaussian count: "
        f"{synthetic_result.get('removed_gaussians')}"
    )

else:
    print(
        "WARNING: synthetic_result is not in globals(); "
        "synthetic artifact checks were skipped."
    )


# ------------------------------------------------------------
# 7. GRAIN LOOKUP ROUND-TRIP CHECK
# ------------------------------------------------------------
GRAIN_BASELINE = memory.grain_lookup(
    "baseline",
    16147,
    include_indices=False,
)

GRAIN_LATEST = memory.grain_lookup(
    "latest",
    16147,
    include_indices=False,
)

assert (
    GRAIN_BASELINE.get("ok")
    is True
), "Baseline grain lookup failed."

assert (
    GRAIN_LATEST.get("ok")
    is True
), "Latest grain lookup failed."

assert (
    GRAIN_BASELINE.get(
        "grain_id"
    )
    == 16147
)

assert (
    GRAIN_LATEST.get(
        "grain_id"
    )
    == 16147
)

# Decoded opacity must be physically bounded.
for record in [
    GRAIN_BASELINE,
    GRAIN_LATEST,
]:
    opacity = record.get(
        "mean_opacity"
    )

    if opacity is not None:
        assert (
            0.0
            <= float(opacity)
            <= 1.0
        ), (
            "Decoded opacity is outside [0,1]: "
            f"{opacity}"
        )

    scales = record.get(
        "mean_scale"
    )

    if scales is not None:
        assert all(
            float(s) > 0
            for s in scales
        ), (
            "Decoded scales must be positive: "
            f"{scales}"
        )


# ------------------------------------------------------------
# 8. REGION SEMANTICS CHECK
# ------------------------------------------------------------
REGION_RESULT = REGION_ANALYZER.detect_regions(
    "baseline"
)

assert (
    REGION_RESULT.get("ok")
    is True
), "detect_regions failed."

assert (
    REGION_RESULT.get(
        "region_count",
        0,
    )
    >= 0
), "Invalid region count."

# Semantic labels must NOT be exposed as verified truth.
assert (
    REGION_RESULT.get(
        "method",
        ""
    )
    != "ground_truth"
), (
    "Region detector incorrectly identifies hypotheses "
    "as ground truth."
)


# ------------------------------------------------------------
# 9. NEGATIVE TEST — UNKNOWN GRAIN
# ------------------------------------------------------------
UNKNOWN_GRAIN_RESULT = execute_tool(
    "grain_lookup",
    {
        "session": "baseline",
        "grain_id": 999999999,
    },
)

assert (
    UNKNOWN_GRAIN_RESULT.get("ok")
    is False
), "Unknown grain ID did not fail safely."

assert (
    UNKNOWN_GRAIN_RESULT.get(
        "error_type"
    )
    == "UNKNOWN_GRAIN_ID"
), (
    "Unknown grain ID returned the wrong error type."
)


# ------------------------------------------------------------
# 10. NEGATIVE TEST — INVALID COORDINATE
# ------------------------------------------------------------
try:
    memory.probe_point(
        "baseline",
        [
            np.nan,
            0.0,
            0.0,
        ],
        radius=0.02,
    )

    raise AssertionError(
        "Non-finite coordinate did not raise."
    )

except ValueError:
    pass


# ------------------------------------------------------------
# 11. NEGATIVE TEST — INVALID RADIUS
# ------------------------------------------------------------
try:
    memory.probe_point(
        "baseline",
        [
            0.0,
            0.0,
            0.0,
        ],
        radius=0.0,
    )

    raise AssertionError(
        "Invalid radius did not raise."
    )

except ValueError:
    pass


# ------------------------------------------------------------
# 12. NEGATIVE TEST — ΔH WITHOUT VALID LATEST
# ------------------------------------------------------------
# Do not destroy the real latest session. Test only the
# typed error by asking for a deliberately missing session.
DELTA_UNAVAILABLE = DAMAGE_ANALYZER.delta_h(
    "baseline",
    "does_not_exist",
)

assert (
    DELTA_UNAVAILABLE.get(
        "ok"
    )
    is False
), "Missing temporal session did not fail safely."

assert (
    DELTA_UNAVAILABLE.get(
        "error_type"
    )
    == "LATEST_SESSION_UNAVAILABLE"
), (
    "Missing temporal session returned wrong error type."
)


# ------------------------------------------------------------
# FINAL DETERMINISTIC REPORT
# ------------------------------------------------------------
print()
print(
    "Synthetic damage validation: PASS"
)

print(
    "Baseline persistent grains:",
    BASELINE_SUMMARY[
        "unique_persistent_grain_count"
    ],
)

print(
    "Baseline Gaussians:",
    BASELINE_SUMMARY[
        "gaussian_count"
    ],
)

print(
    "Synthetic latest Gaussians:",
    LATEST_SUMMARY[
        "gaussian_count"
    ],
)

print(
    "Missing persistent grains:",
    DETECT_RESULT[
        "total_missing_grains"
    ],
)

print(
    "Partially reduced grains:",
    DETECT_RESULT[
        "partially_reduced_grains"
    ],
)

print(
    "Affected grains:",
    DETECT_RESULT[
        "affected_grain_count"
    ],
)

print(
    "Damage zones:",
    DETECT_RESULT[
        "damage_zones_count"
    ],
)

print(
    "Flagged ΔH cells:",
    DELTA_H_RESULT.get(
        "flagged_damage_cells",
        0,
    ),
)

print()
print(
    "Deterministic regression checks: PASS"
)

DETERMINISTIC PIPELINE VALIDATION

Synthetic damage validation: PASS
Baseline persistent grains: 35731
Baseline Gaussians: 194018
Synthetic latest Gaussians: 182928
Missing persistent grains: 2117
Partially reduced grains: 160
Affected grains: 2277
Damage zones: 2
Flagged ΔH cells: 122

Deterministic regression checks: PASS
